# Unitree Go1：揺れる甲板の上を60秒、転倒せずに歩く（v6・速度で切り替える補正モデル）

**Google Colab ProでGPU「L4」を選び、メニューの「ランタイム → すべてのセルを実行」で、評価・比較動画まで再現できます（既定では約1時間）。**
Google Driveへの接続の承認が1回必要です。**Driveに2つのファイルを置いてください**（下の「準備」）。

## 何をするノートブックか
- 公開されている四脚ロボット歩行方策 **WTW（Walk These Ways）** を、物理シミュレータ **MuJoCo** 上のUnitree Go1に載せ、
  船の甲板のように揺れる床（ロール・ピッチ・上下動）の上を一定速度で60秒歩かせます。WTWの重みは変更せず、
  その出力（12関節の目標角）に小さな補正を足す**補正モデル**を、強化学習の **PPO（Proximal Policy Optimization）** で学習します。
- 目標は、**評価用の波（seed 1000〜1004）× 指令速度0.10／0.20／0.30 m/s の15条件で、60秒歩き切る条件を、最初のノートブックより増やすこと**です。

## v6の考え方：指令速度で補正モデルを切り替える
| 指令速度 | 使う補正モデル | 理由 |
|---|---|---|
| **0.15 m/s以上**（評価では0.20・0.30） | **最初のノートブックのモデルをそのまま** | この速度では最初のモデルが強く、追加学習するとかえって完走が減りました |
| **0.15 m/s未満**（評価では0.10） | **低速専用モデル**（最初のモデルから追加学習） | 最初のモデルは0.10 m/sで5条件中2条件しか完走できず、ここに改善の余地があります |

指令速度は1回の走行の間ずっと一定なので、どちらのモデルを使うかは走り出す前に決まり、走行中に切り替わることはありません。
このように運転条件ごとに制御器を使い分ける方法は、制御工学で**ゲインスケジューリング**と呼ばれる一般的な手法です。

## 準備：Google Driveに置く2つのファイル
| ファイル | Driveの置き場所（既定） | 内容 |
|---|---|---|
| 最初のノートブックの学習済みモデル | `MyDrive/ship_robot_runs/experiment04_multispeed/level_04/policy.pkl` | 最初のノートブック（`unitree_go1_multispeed.ipynb`）を実行すると、ここに保存されます。0.20・0.30 m/sではこれをそのまま使います |
| 低速専用モデル | `MyDrive/ship_robot_runs/v6_trained/low_speed_specialist.pkl` | 配布物の`trained/low_speed_specialist.pkl`。著者がこのノートブックと同じ手順で学習し、検証の波だけで選んだモデルです |

場所が違う場合は、下の`ORIGINAL_POLICY`・`SPECIALIST_POLICY`を書き換えてください。
- 最初のモデル：著者が使ったファイルのSHA-256は`a89b2c5a0d850861…`です。最初のノートブックを実行し直したモデルは学習の揺らぎで別物になるため、一致しない場合は警告を表示します（実行は続けます。結果の数値は著者のものと変わることがあります）。
- 低速専用モデル：SHA-256が一致しない場合は停止します（Pythonのpickleファイルは、信頼できる入手元のものだけ読み込んでください）。

### なぜ低速専用モデルを同梱するのか
低速専用モデルの学習（PPO）は、GPUの並列計算の順序によって実行ごとに少し結果が変わります。著者が同じ手順を3つの乱数seedで実行すると、
評価15条件の完走は**11・10・10**でした（下の表）。手順どおり「検証の波で最良のもの」を選ぶとseed 5（11完走）が選ばれますが、
Colabで学習し直すと、同じモデルは得られません。そこで、**著者が選んだ低速専用モデルを同梱し、Colab L4では評価・対照・動画を実行**します。
v6の評価では、0.20・0.30 m/sの走行を2回計算しません。指令速度は1回の走行の間ずっと一定なので、この速度でのv6はv0そのものです。そこで、同じ評価で走らせたv0の走行をそのままv6の結果とし、0.10 m/sだけ低速専用モデルで走らせて組み合わせます（`test_trained/joined.json`に記録）。GPUは実行ごとに計算の方式や足し算の順序が変わることがあり、同じ制御器を2回走らせると、際どい条件の完走・転倒がわずかな差で入れ替わることがあるためです（著者のColabでの最初の実行では、0.20 m/sのseed 1000で、v0は完走、v6として走らせ直したv0のネットワークは11.6秒で転倒しました。この条件はv0でも、開始位置を9通りずらすと完走は2通りだけです）。CPUとGPUでも計算の細部が違うため、際どい条件では結果が異なることがあります。
学習から再現したい場合は、`TRAIN_SPECIALIST = True`にすると、このノートブックの中で3つの乱数seedで学習し、検証の波で1つを選びます（約1.5時間増）。

## seedの使い分け（評価用の波は学習にも選択にも使いません）
| 用途 | seed（波の番号） | 説明 |
|---|---|---|
| **評価（テスト）** | **1000〜1004** | 最後に1回だけ使います。最初のノートブックと同じ波・同じ手順です |
| 検証（低速専用モデルの選択） | 5000番台・7000番台の30個 | 0.10 m/sで「難しいが物理的に歩ける」波を自動で選びます（下の説明） |
| 学習 | 10000以上の乱数 | エピソードごとに新しい波を作ります |

## 実行の目安
- Colab L4（既定）：準備 約10分、検証の波の確認 約5分、評価 約15分、静止甲板の対照 約5分、動画 約25分。合計 約1時間（見込み）。
- `TRAIN_SPECIALIST = True`の場合：低速専用モデルの学習（3つの乱数seed×200万ステップ）と選択で、さらに約1.5時間（見込み）。
- CPUだけのLinux（例：WSL2）でも実行できます（`ALLOW_CPU=True`）。評価だけで約2時間、学習は1通り約1.5時間かかります。

## 著者の事前検証（CPU、WSL2上のLinux）
評価：波seed 1000〜1004×指令速度0.10／0.20／0.30 m/s、各60秒、途中リセットなし。

| | 公開方策のみ（学習前） | 最初のノートブックのモデル | **v6（速度で切り替え）** |
|---|---|---|---|
| 評価15条件の完走 | 6 | 10 | **11** |
| うち0.10 m/s（5条件） | 2 | 2 | **3**（seed 1001を新たに完走） |
| うち0.20・0.30 m/s（10条件） | 4 | 8 | 8（最初のモデルと全く同じ） |
| 静止甲板15条件の完走 | 15 | 15 | 15 |
| 検証の波（0.10 m/s、30条件）の完走 | 9 | 10 | **21** |

- v6が最初のモデルより悪くなった条件はありません（転倒→完走が1条件、完走→転倒が0条件）。
- 開始位置を前後±4 cm・左右±2 cmずらした9通りで同じ評価をすると、完走数の期待値は最初のモデル9.22、v6 10.22（seed 1003を除く12条件）でした。v6が新たに完走したseed 1001（0.10 m/s）は9通りすべてで完走し、偶然ではありません。
- 速度追従：0.20・0.30 m/sは最初のモデルと同一です。0.10 m/sの揺れる甲板では、v6は速度の誤差（RMSE）が0.137 m/s（最初のモデル0.111 m/s）で、指令よりやや速く歩きます。静止甲板では0.013／0.014／0.019 m/s（最初のモデル0.011／0.014／0.019 m/s）です。
- seed 1003は約4.8秒で甲板が重力加速度より速く落ち、足が甲板から離れるため、どのモデルも完走できません（実質的な上限は12/15）。

### 低速専用モデルの学習の揺らぎ（同じ手順、乱数seedだけを変更）
| 学習seed | 検証の波 完走 / 30 | 評価15条件の完走（v6として） |
|---|---|---|
| **5（同梱したモデル）** | **21** | **11** |
| 6 | 19 | 10 |
| 7 | 18 | 10 |

手順どおり検証の波の成績で選ぶとseed 5が選ばれます。ほかの2つは評価で最初のモデルと同じ10完走でした（0.10 m/sのseed 1001は救えるが、seed 1002を失う）。
学習し直すと結果が変わりうるため、既定では同梱した低速専用モデルを使います。

## 1 実験設定
通常は変更不要です。設定を変えて再実行するときは、`OUTPUT_NAME`を別の名前にしてください（結果の混在を防ぐため、設定が変わると停止します）。

In [ ]:
from pathlib import Path
import sys, os, json

OUTPUT_NAME = 'experiment62_ship_v6'
ALLOW_CPU = False            # GPUがない環境（WSL2など）で実行する場合だけTrue
# 最初のノートブックの学習済みモデル（Colabでは下の既定の場所。Colab以外ではファイルのパス）
ORIGINAL_POLICY = '/content/drive/MyDrive/ship_robot_runs/experiment04_multispeed/level_04/policy.pkl'
ORIGINAL_SHA256 = 'a89b2c5a0d850861d925361f472d04fc2ee7f810d776b7908fe1793eb2dc872d'   # 著者が使ったファイル（一致しなくても実行は続けます）
# 低速専用モデル：False＝著者が学習・選択したモデル（配布物の trained/low_speed_specialist.pkl）を使う
#                 True ＝このノートブックで学習し直す（約1.5時間増。学習の揺らぎで結果は変わります）
TRAIN_SPECIALIST = False
SPECIALIST_POLICY = '/content/drive/MyDrive/ship_robot_runs/v6_trained/low_speed_specialist.pkl'
SPECIALIST_SHA256 = '016d8f395482cbb786bdb5aa2f605ad9b6b9117f4302ad4592b3bd74e6137744'

CONFIG = {
    'test_condition': [18.0, 12.0, 0.10],          # 甲板の揺れ：ロール18度、ピッチ12度、上下0.10 m（最初のノートブックの最終段階と同じ）
    'speeds': [0.10, 0.20, 0.30],                   # 指令速度 [m/s]
    'test_seeds': [1000, 1001, 1002, 1003, 1004],   # 評価用の波（最後に1回だけ使用）
    'seconds': 60.0,
    'switch_speed_m_s': 0.15,                       # これ未満の指令速度で低速専用モデルを使う
    # 低速専用モデル（最初のモデルから追加学習）。各値の意味は下の「6」の説明を参照
    'specialist': {"seeds": [5, 6, 7], "steps": 2000000, "envs": 32, "episode_length": 1500, "lr": 0.0001, "checkpoints": 8, "validation_speed": 0.1, "validation": {"hard_count": 20, "hard_first": 5000, "hard_lo": 0.75, "hard_hi": 1.0, "early_count": 10, "early_first": 7000, "early_window_s": 10, "early_lo": 0.8, "early_hi": 1.0}, "env_overrides": {"speed_min": 0.1, "speed_max": 0.15, "start_wide_prob": 0.15, "amp_min_frac": 0.0, "full_amp_prob": 0.6, "calm_prob": 0.1, "early_bonus": 1.0, "alive_bonus": 0.0, "fall_penalty": 5.0, "lateral_weight": 0.15, "margin_weight": 3.0}},
    'flat_control': True,                           # 静止甲板での対照評価
    'video': True,
}
v = CONFIG['specialist']['validation']
assert v['hard_first'] > 1004 and v['early_first'] > 1004, '検証の波は評価用seedより後の番号から選ぶ'
print(CONFIG)

## 2 保存先と最初のモデルの確認
Colabでは結果をGoogle Driveの`MyDrive/ship_robot_runs/`に保存します（途中で切断されても、完了済みの段階は再実行時にスキップされます）。
Colab以外では、このノートブックと同じフォルダーの`runs/`に保存します。

In [ ]:
import hashlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = f'/content/drive/MyDrive/ship_robot_runs/{OUTPUT_NAME}'
    WORK = Path('/content/unitree_go1_ship_v6')
else:
    OUTPUT = str(Path('runs', OUTPUT_NAME).resolve())
    WORK = Path('unitree_go1_ship_v6_src').resolve()
ORIGINAL_POLICY = str(Path(ORIGINAL_POLICY).resolve())   # 子プロセスは別のフォルダーで動くため絶対パスにする
assert Path(ORIGINAL_POLICY).exists(), f'最初のノートブックの学習済みモデルが見つかりません: {ORIGINAL_POLICY}'
actual = hashlib.sha256(Path(ORIGINAL_POLICY).read_bytes()).hexdigest()
print('最初のモデル SHA-256:', actual)
if actual != ORIGINAL_SHA256:
    print('注意：著者が使ったモデルと異なります。結果の数値は著者のものと変わることがあります。')
CONFIG.update(output=OUTPUT, assets=OUTPUT + '/public_assets', allow_cpu=ALLOW_CPU, original_policy=ORIGINAL_POLICY)
if not TRAIN_SPECIALIST:
    SPECIALIST_POLICY = str(Path(SPECIALIST_POLICY).resolve())
    assert Path(SPECIALIST_POLICY).exists(), f'低速専用モデルが見つかりません: {SPECIALIST_POLICY}'
    actual = hashlib.sha256(Path(SPECIALIST_POLICY).read_bytes()).hexdigest()
    print('低速専用モデル SHA-256:', actual)
    assert actual == SPECIALIST_SHA256, '配布した低速専用モデルと一致しません（信頼できないpickleは読み込まないでください）'
    CONFIG['specialist_policy'] = SPECIALIST_POLICY
Path(OUTPUT).mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
WORK_STR = str(WORK)
print('結果:', OUTPUT, '\nコード:', WORK)

## 3 コードをファイルごとに保存
以下の`%%writefile`セルは、それぞれ1つのPythonファイルを保存するだけです（学習はまだ始まりません）。
★はv6で新規または変更したファイル、無印は最初のノートブックと同一のファイルです。

### `model.py`
Go1と40 mの甲板（MuJoCoモデル）の組み立て、関節の対応、WTWの観測の定義。最初のノートブックと同一。

In [ ]:
%%writefile $WORK_STR/model.py
"""Pinned Menagerie Go1, torque PD as in WTW hardware deployment."""
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import mujoco

LEGS = ['FL','FR','RL','RR']
NAMES = [f'{leg}_{axis}_joint' for leg in LEGS for axis in ['hip','thigh','calf']]
HOME = np.array([.1,.8,-1.5,-.1,.8,-1.5,.1,1.,-1.5,-.1,1.,-1.5],np.float32)
SCALE = np.tile([.125,.25,.25],4).astype(np.float32)

def build(asset_dir, out):
    src = Path(asset_dir)/'unitree_go1'
    root = ET.parse(src/'go1.xml').getroot()
    root.find('compiler').set('meshdir',str((src/'assets').resolve()))
    opt=root.find('option'); opt.attrib.update(timestep='.0025',cone='pyramidal',solver='Newton',iterations='10',ls_iterations='10',impratio='1')
    # WTW hardware applies explicit 20/0.5 PD. Remove Menagerie position servos
    # and extra joint damping, which otherwise would apply PD twice.
    for joint in root.findall('.//default//joint'):
        joint.attrib.update(damping='0',frictionloss='0',armature='.01')
    for geom in root.findall('.//default//geom'):
        if geom.get('type')=='cylinder': geom.set('type','capsule')
        if geom.get('condim'): geom.attrib.update(condim='3',solimp='.9 .95 .001',solref='.02 1')
    for geom in root.findall('.//worldbody//geom'):
        if geom.get('type')=='cylinder': geom.set('type','capsule')
        if geom.get('class')!='visual': geom.attrib.update(contype='1',conaffinity='2')
    world=root.find('worldbody')
    robot=world.find("body[@name='trunk']"); robot.set('pos','-12 0 1.24')
    light=world.find('light'); light.attrib.update(mode='fixed',pos='0 -2 5'); light.attrib.pop('target',None)
    ET.SubElement(world,'geom',type='plane',size='10 10 .1',rgba='.1 .23 .32 1',contype='0',conaffinity='0')
    heave=ET.Element('body',name='heave',pos='0 0 .8')
    ET.SubElement(heave,'joint',name='heave',type='slide',axis='0 0 1',damping='0')
    ET.SubElement(heave,'inertial',mass='1',pos='0 0 0',diaginertia='1 1 1')
    roll=ET.SubElement(heave,'body',name='roll')
    ET.SubElement(roll,'joint',name='roll',axis='1 0 0',damping='0')
    ET.SubElement(roll,'inertial',mass='1',pos='0 0 0',diaginertia='1 1 1')
    deck=ET.SubElement(roll,'body',name='deck')
    ET.SubElement(deck,'joint',name='pitch',axis='0 1 0',damping='0')
    ET.SubElement(deck,'geom',name='deck',type='box',size='20 2 .1',mass='100',contype='2',conaffinity='1',condim='3',friction='1 .005 .0001',rgba='.35 .45 .5 1')
    for x in np.arange(-19.5,20.,.5):
        ET.SubElement(deck,'geom',type='box',pos=f'{x} 0 .102',size='.008 1.9 .002',mass='0',contype='0',conaffinity='0',rgba='.8 .8 .7 1')
    world.insert(1,heave)
    root.remove(root.find('actuator')); act=ET.SubElement(root,'actuator')
    for n in ['heave','roll','pitch']: ET.SubElement(act,'motor',name=n,joint=n)
    for n in NAMES:
        lim=35.55 if 'calf' in n else 23.7
        ET.SubElement(act,'motor',name=n,joint=n,ctrlrange=f'{-lim} {lim}')
    key=root.find('keyframe')
    if key is not None: root.remove(key)
    visual=root.find('visual')
    if visual is None: visual=ET.SubElement(root,'visual')
    ET.SubElement(visual,'global',offwidth='960',offheight='540')
    ET.SubElement(visual,'headlight',ambient='.5 .5 .5',diffuse='.8 .8 .8',specular='.2 .2 .2')
    out=Path(out);out.parent.mkdir(parents=True,exist_ok=True)
    ET.ElementTree(root).write(out,encoding='unicode')
    return out

def mapping(m):
    # Menagerie order is FR, FL, RR, RL; network order is FL, FR, RL, RR.
    ids=np.array([m.joint(n).id for n in NAMES])
    return m.jnt_qposadr[ids],m.jnt_dofadr[ids]

def observation(q,v,rotation,action,previous,phase,speed,xp=np,side=0.,yaw=0.):
    command=xp.array([speed*2,side*2,yaw*.25,0,3,0.5,0,0,0.5,0.08*.15,0,0,.25,.4,0])
    clocks=xp.sin(2*xp.pi*(phase+xp.array([.5,0,0,.5])))
    return xp.clip(xp.concatenate([rotation.T@xp.array([0.,0.,-1.]),command,
        q-xp.asarray(HOME),v*.05,action,previous,clocks]),-100,100)

### `policy.py`
WTW方策（重み固定）の推論。最初のノートブックと同一。

In [ ]:
%%writefile $WORK_STR/policy.py
"""ELU inference, unchanged WTW adaptation module + actor weights."""
import numpy as np
import jax.numpy as jp
import jax
class FrozenPolicy:
    def __init__(self,path):
        with np.load(path) as z:self.w={k:jp.array(z[k]) for k in z.files}
    def net(self,x,prefix,layers):
        for i in layers:
            x=x@self.w[f'{prefix}.{i}.weight'].T+self.w[f'{prefix}.{i}.bias']
            if i!=layers[-1]:x=jax.nn.elu(x)
        return x
    def __call__(self,h):
        latent=self.net(h,'adaptation_module',[0,2,4])
        return self.net(jp.concatenate([h,latent],axis=-1),'actor_body',[0,2,4,6])

### `ship_env.py`
★ 甲板の物理駆動、波の生成、学習時の波・開始位置・速度の抽選、報酬。評価時の動作は最初のノートブックと同一です（公開方策のみの15条件で転倒時刻が一致することを確認）。

In [ ]:
%%writefile $WORK_STR/ship_env.py
"""WTW frozen Go1 gait + learned joint-target residual, physical moving deck (v5).

Evaluation (random_start=False) is identical to the original multispeed notebook:
the waves of test seed s come from PRNGKey(s), start x=-12 m, y=0.
Training (random_start=True) draws an integer wave id >= wave_id_min and builds the
waves with exactly the same generator, so training never sees test/validation ids.
"""
import os
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jp
from flax import struct
from brax.envs.base import Env,State
from model import build,mapping,HOME,SCALE,observation
from policy import FrozenPolicy

METRICS=['velocity','height','slip_proxy','reward','fall']

@dataclass(frozen=True)
class Config:
    roll_deg:float=2.
    pitch_deg:float=1.4
    heave_m:float=.02
    speed:float=.1
    random_speed:bool=False
    random_start:bool=False
    speed_min:float=.10
    speed_max:float=.30
    # ---- training-only distribution (ignored when random_start=False) ----
    start_min:float=-12.5        # start x [m]; evaluation always starts at -12
    start_max:float=-11.5
    start_y:float=.2             # half-range of start y [m]
    start_wide_prob:float=0.     # ... or, with this probability, anywhere in [start_wide_min, start_wide_max]
    start_wide_min:float=-12.
    start_wide_max:float=10.
    low_speed_prob:float=0.      # with this probability the speed command is drawn from [speed_min, low_speed_max]
    low_speed_max:float=.15
    wave_id_min:int=10_000       # training wave ids are >= this (test 1000-1004, validation 5000-5019)
    amp_min_frac:float=0.        # per-episode amplitude scale drawn from [amp_min_frac, 1]
    full_amp_prob:float=.5       # ... or exactly 1 with this probability
    calm_prob:float=.1           # ... or 0 (static deck) with this probability
    wave_candidates:int=4        # draw this many wave ids and pick one by hardness
    hard_bonus:float=.7          # logit bonus for hard but feasible waves (0.6 <= M < 1.05)
    infeasible_penalty:float=1.5 # logit penalty for waves with M >= 1.05 (deck falls faster than g)
    early_bonus:float=0.         # extra logit for waves whose deck drop is already hard in the first early_seconds
    early_seconds:float=10.      # (the 5 s wave ramp and the first steps, where the robot is still near the deck end)
    # ---- reward / controller ----
    residual_rad:float=.12
    alive_bonus:float=1.
    fall_penalty:float=10.
    lateral_weight:float=1.
    margin_weight:float=0.       # penalty as the trunk nears the 0.26 m fall line (0 at 0.31 m, full at 0.26 m)
    sim_dt:float=.0025
    substeps:int=8

@struct.dataclass
class Pipeline:
    data:object
    waves:object
    history:object
    last:object
    previous:object
    residual:object
    speed:object
    amplitude:object

def motion(t,waves,amplitude):
    s=jp.clip(t/5.,0,1); e=10*s**3-15*s**4+6*s**5
    ed=jp.where(t<5,(30*s**2-60*s**3+30*s**4)/5,0.)
    angle=waves[...,0]*t+waves[...,1]
    raw=jp.mean(jp.sin(angle),axis=-1);rd=jp.mean(waves[...,0]*jp.cos(angle),axis=-1)
    return amplitude*e*raw,amplitude*(ed*raw+e*rd)

def waves_from_key(rng):
    """Original generator: 3 axes x 3 sines, periods 2-6 s, random phases."""
    k1,k2=jax.random.split(rng)
    periods=jax.random.uniform(k1,(3,3),minval=2.,maxval=6.)
    phases=jax.random.uniform(k2,(3,3),minval=-jp.pi,maxval=jp.pi)
    return jp.stack([2*jp.pi/periods,phases],axis=-1)

def waves_from_id(wave_id):
    """Same waves as the evaluation of seed `wave_id` (reset key split exactly as in evaluation)."""
    return waves_from_key(jax.random.split(jax.random.PRNGKey(wave_id),3)[0])

def hardness(waves,amplitude,speed,x0,seconds=60.,dt=.05):
    """M = largest downward acceleration of the deck surface under the robot's nominal path, in g.
    M >= 1 means the deck drops faster than gravity, so the feet cannot stay in contact."""
    t=jp.arange(0.,seconds,dt)
    q=jax.vmap(lambda s:motion(s,waves,amplitude)[0])(t)
    h,r,p=q[:,0],q[:,1],q[:,2];x=x0+speed*t
    z=h+jp.cos(r)*(-jp.sin(p)*x+jp.cos(p)*.1)
    acc=(z[2:]-2*z[1:-1]+z[:-2])/dt**2
    return -jp.min(acc)/9.81

class ShipEnv(Env):
    def __init__(self,config=Config(),assets='assets'):
        if not np.isclose(config.sim_dt*config.substeps,.02):
            raise ValueError('WTW policy requires a 20 ms control period')
        self.config=config; self.assets=Path(assets).resolve()
        # One scene file per process: parallel evaluations must not overwrite each other's file.
        self.xml=build(self.assets,self.assets/f'ship_scene_{os.getpid()}.xml')
        self.mj_model=mujoco.MjModel.from_xml_path(str(self.xml))
        self.mj_model.opt.timestep=config.sim_dt
        self.sys=mjx.put_model(self.mj_model)
        self.qi,self.vi=map(jp.array,mapping(self.mj_model))
        self.base_id=self.mj_model.body('trunk').id;self.deck_id=self.mj_model.body('deck').id
        self.foot_ids=jp.array([self.mj_model.geom(n).id for n in ['FL','FR','RL','RR']])
        self.q0=jp.array(self.mj_model.qpos0).at[self.qi].set(HOME)
        native=mujoco.MjData(self.mj_model)
        native.qpos[:]=np.asarray(self.q0)
        mujoco.mj_forward(self.mj_model,native)
        self._data=mjx.put_data(self.mj_model,native)
        robot=self.mj_model.body_rootid==self.base_id
        self.body_robot=jp.array(robot[:,None])
        self.geom_robot=jp.array(robot[self.mj_model.geom_bodyid,None])
        self.frozen=FrozenPolicy(self.assets/'policy.npz')
        self.amplitude=jp.array([config.heave_m,np.deg2rad(config.roll_deg),np.deg2rad(config.pitch_deg)])
        self.robot_mass=float(self.mj_model.body_subtreemass[self.base_id])
        self.limits=jp.array(self.mj_model.actuator_ctrlrange[3:,1])
    @property
    def dt(self):return self.config.sim_dt*self.config.substeps
    @property
    def action_size(self):return 12
    @property
    def observation_size(self):return 94
    @property
    def backend(self):return 'mjx'
    def relative(self,d):
        rot=d.xmat[self.deck_id];delta=d.xpos[self.base_id]-d.xpos[self.deck_id]
        roll=d.qpos[1];omega=jp.array([d.qvel[1],jp.cos(roll)*d.qvel[2],jp.sin(roll)*d.qvel[2]])
        deck_v=jp.array([0.,0.,d.qvel[0]])+jp.cross(omega,delta)
        return rot.T@delta,rot.T@(d.qvel[3:6]-deck_v)
    def baseobs(self,d,last,previous,speed):
        pos,_=self.relative(d); rot=d.xmat[self.base_id]
        yaw=jp.arctan2(rot[1,0],rot[0,0])
        return observation(d.qpos[self.qi],d.qvel[self.vi],rot,last,previous,
                           jp.mod(d.time*3.,1.),speed,jp,
                           jp.clip(-.4*pos[1],-.2,.2),jp.clip(-2*yaw,-.5,.5))
    def observe(self,p):
        d=p.data;_,vel=self.relative(d)
        up=d.xmat[self.base_id].T@d.xmat[self.deck_id][:,2]
        return jp.concatenate([p.history[-70:],vel,up,d.qpos[:3],d.qvel[:3],p.residual])
    def training_waves(self,rng,speed,x0):
        """Pick one of several fresh wave ids, favouring hard-but-feasible ones; random amplitude scale."""
        c=self.config;k_id,k_pick,k_amp,k_full=jax.random.split(rng,4)
        ids=jax.random.randint(k_id,(c.wave_candidates,),c.wave_id_min,2**31-1)
        frac=jax.random.uniform(k_amp,(),minval=c.amp_min_frac,maxval=1.)
        u=jax.random.uniform(k_full,())
        frac=jp.where(u<c.full_amp_prob,1.,jp.where(u<c.full_amp_prob+c.calm_prob,0.,frac))
        amp=self.amplitude*frac
        cand=jax.vmap(waves_from_id)(ids)
        m=jax.vmap(lambda w:hardness(w,amp,speed,x0))(cand)
        me=jax.vmap(lambda w:hardness(w,amp,speed,x0,seconds=c.early_seconds))(cand)
        logits=c.hard_bonus*((m>=.6)&(m<1.05))-c.infeasible_penalty*(m>=1.05)+c.early_bonus*((me>=.7)&(me<1.05))
        return cand[jax.random.categorical(k_pick,logits)],amp
    def reset(self,rng):
        c=self.config;kw,kv,kx=jax.random.split(rng,3)
        if c.random_speed:
            kl=jax.random.fold_in(kv,1)
            hi=jp.where(jax.random.uniform(kl,())<c.low_speed_prob,c.low_speed_max,c.speed_max)
            speed=jax.random.uniform(kv,(),minval=c.speed_min,maxval=hi)
        else:
            speed=jp.asarray(c.speed,jp.float32)
        if c.random_start:
            ky=jax.random.fold_in(kx,1);kw2=jax.random.fold_in(kx,2)
            wide=jax.random.uniform(kw2,())<c.start_wide_prob
            x0=jp.where(wide,jax.random.uniform(jax.random.fold_in(kx,3),(),minval=c.start_wide_min,maxval=c.start_wide_max),
                        jax.random.uniform(kx,(),minval=c.start_min,maxval=c.start_max))
            q0=self.q0.at[3].set(x0)
            q0=q0.at[4].set(jax.random.uniform(ky,(),minval=-c.start_y,maxval=c.start_y))
            waves,amp=self.training_waves(kw,speed,q0[3])
        else:
            q0=self.q0;waves=waves_from_key(kw);amp=self.amplitude
        shift=jp.array([q0[3]-self.q0[3],q0[4]-self.q0[4],0.])
        d=self._data.replace(qpos=q0,
            xpos=self._data.xpos+self.body_robot*shift,
            geom_xpos=self._data.geom_xpos+self.geom_robot*shift)
        zero=jp.zeros(12);obs=self.baseobs(d,zero,zero,speed)
        h=jp.concatenate([jp.zeros(2030),obs])
        p=Pipeline(d,waves,h,zero,zero,zero,speed,amp)
        z=jp.array(0.,jp.float32)
        return State(p,self.observe(p),z,z,{n:z for n in METRICS},{})
    def control(self,d,waves,amplitude,qtarget):
        target,tv=motion(d.time,waves,amplitude)
        deck=jp.array([30000.,40000.,2666667.])*(target-d.qpos[:3])+jp.array([1600.,4000.,266667.])*(tv-d.qvel[:3])+d.qfrc_bias[:3]
        deck=deck.at[0].add(self.robot_mass*9.81)
        tau=jp.clip(20*(qtarget-d.qpos[self.qi])-.5*d.qvel[self.vi],-self.limits,self.limits)
        return jp.concatenate([deck,tau])
    def feet_relative(self,d):return (d.geom_xpos[self.foot_ids]-d.xpos[self.deck_id])@d.xmat[self.deck_id]
    def step(self,state,action):
        c=self.config;old=state.pipeline_state; action=jp.clip(action,-1,1)
        base=jp.clip(self.frozen(old.history),-10,10)
        # Residual is bounded in joint-target radians; the base policy never changes.
        total=jp.clip(base+c.residual_rad*action/jp.array(SCALE),-10,10)
        qtarget=jp.array(HOME)+jp.array(SCALE)*total
        def substep(d,_):return mjx.step(self.sys,d.replace(ctrl=self.control(d,old.waves,old.amplitude,qtarget))),None
        d,_=jax.lax.scan(substep,old.data,None,length=c.substeps)
        d=mjx.forward(self.sys,d)
        obs=self.baseobs(d,total,old.last,old.speed);h=jp.concatenate([old.history[70:],obs])
        p=Pipeline(d,old.waves,h,total,old.last,action,old.speed,old.amplitude)
        pos,vel=self.relative(d);upright=d.xmat[self.base_id][2,2]
        feet=self.feet_relative(d);oldfeet=self.feet_relative(old.data)
        near=(feet[:,2]<.14)&(oldfeet[:,2]<.14)
        slip=jp.mean(near*jp.sum(((feet-oldfeet)/self.dt)[:,:2]**2,axis=1))
        track=jp.exp(-((vel[0]-old.speed)**2+vel[1]**2)/.01)
        reward=(2*track+upright+c.alive_bonus-10*(pos[2]-.37)**2-.1*jp.sum(d.qvel[6:9]**2)
                -.3*slip-c.lateral_weight*pos[1]**2-.0002*jp.sum(d.ctrl[3:]**2)
                -.02*jp.sum(action**2)-.02*jp.sum((action-old.residual)**2)
                -c.margin_weight*jp.clip((.31-pos[2])/.05,0.,1.))
        bad=(pos[2]<.26)|(upright<.45)|(jp.abs(pos[0])>19.5)|(jp.abs(pos[1])>1.5)|~jp.all(jp.isfinite(d.qpos))|~jp.all(jp.isfinite(d.qvel))
        done=bad.astype(jp.float32);reward=jp.nan_to_num(reward,nan=-10.)-c.fall_penalty*done
        metrics={'velocity':vel[0],'height':pos[2]-.1,'slip_proxy':slip,'reward':reward,'fall':done}
        return state.replace(pipeline_state=p,obs=self.observe(p),reward=reward,done=done,metrics=metrics)
    def write_xml(self,path):Path(path).write_text(self.xml.read_text(),encoding='utf-8')

### `reset_wrapper.py`
エピソード終了ごとに条件を抽選し直す処理。最初のノートブックと同一。

In [ ]:
%%writefile $WORK_STR/reset_wrapper.py
"""Resample commands and waves at EVERY episode boundary.

Brax's stock AutoResetWrapper restores the first state, so it would repeat
the initial per-environment command. This wrapper draws a new reset key.
"""
import jax
import jax.numpy as jp
from brax.envs.base import Wrapper
from brax.envs.wrappers.training import VmapWrapper,EpisodeWrapper

class ResamplingAutoReset(Wrapper):
    def reset(self,rng):
        split=jax.vmap(lambda k:jax.random.split(k,2))(rng)
        state=self.env.reset(split[:,0])
        state.info['reset_rng']=split[:,1]
        return state
    def step(self,state,action):
        state.info['steps']=jp.where(state.done,0,state.info['steps'])
        state=state.replace(done=jp.zeros_like(state.done))
        state=self.env.step(state,action)
        split=jax.vmap(lambda k:jax.random.split(k,2))(state.info['reset_rng'])
        def fresh(_):
            reset=self.env.reset(split[:,0])
            return reset.pipeline_state,reset.obs
        new_pipeline,new_obs=jax.lax.cond(jp.any(state.done),fresh,
            lambda _:(state.pipeline_state,state.obs),operand=None)
        def select(new,old):
            mask=state.done.reshape(state.done.shape+(1,)*(old.ndim-state.done.ndim))
            return jp.where(mask,new,old)
        pipeline=jax.tree.map(select,new_pipeline,state.pipeline_state)
        obs=jax.tree.map(select,new_obs,state.obs)
        state.info['reset_rng']=jp.where(state.done[:,None],split[:,1],state.info['reset_rng'])
        return state.replace(pipeline_state=pipeline,obs=obs)

def wrap(env,episode_length=1000,action_repeat=1,randomization_fn=None):
    if randomization_fn is not None:raise ValueError('This task randomizes within reset, not model parameters')
    return ResamplingAutoReset(EpisodeWrapper(VmapWrapper(env),episode_length,action_repeat))

### `train.py`
★ PPOで補正モデルを学習（最初のモデルからの追加学習に対応）。途中のモデルを保存します。

In [ ]:
%%writefile $WORK_STR/train.py
"""PPO for the residual (v5). Every intermediate policy is kept for validation-based selection.
Checkpoints are trusted local Python pickle files."""
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
import argparse
from dataclasses import asdict
import json
import importlib.metadata
import pickle
from pathlib import Path
import time
import jax
import jax.numpy as jp
from brax.training.agents.ppo import networks, train as ppo
from ship_env import Config, ShipEnv
from reset_wrapper import wrap as training_wrap

TASK = 'ship-residual-v5'


def NETWORK(observation_size, action_size, preprocess_observations_fn=lambda x, _: x, **kwargs):
    # Early running variances can be tiny. Bound normalized inputs to avoid
    # extreme critic predictions during the first few updates.
    def preprocess(obs, params):
        return jp.clip(preprocess_observations_fn(obs, params), -10., 10.)
    return networks.make_ppo_networks(observation_size, action_size,
        preprocess_observations_fn=preprocess,
        policy_hidden_layer_sizes=(128,128,128),
        value_hidden_layer_sizes=(128,128,128), **kwargs)


def load(path):
    # Never load a pickle obtained from an untrusted source.
    with open(path, 'rb') as f:
        return pickle.load(f)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--assets', default='assets')
    p.add_argument('--out', required=True)
    p.add_argument('--steps', type=int, default=6_000_000)
    p.add_argument('--envs', type=int, default=32)
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--checkpoints', type=int, default=12, help='policies saved during training (validation candidates)')
    p.add_argument('--episode-length', type=int, default=3000)
    p.add_argument('--config-json', default='{}', help='overrides of ship_env.Config fields')
    p.add_argument('--lr', type=float, default=3e-4, help='PPO learning rate')
    p.add_argument('--restore', help='trusted policy pickle to start from (this task, or the original multispeed notebook)')
    p.add_argument('--allow-cpu', action='store_true')
    p.add_argument('--tiny', action='store_true', help='API smoke test, not training')
    a = p.parse_args()
    if not a.allow_cpu and jax.default_backend() != 'gpu':
        raise RuntimeError('GPU is not active. Select Colab L4, restart runtime, check jax.devices().')
    if not a.tiny and (a.envs > 256 or 256 % a.envs):
        p.error('--envs must divide 256: use 8, 16, 32, 64, 128 or 256')
    over = json.loads(a.config_json)
    cfg = Config(**{'roll_deg': 18., 'pitch_deg': 12., 'heave_m': .10,
                    'random_speed': True, 'random_start': True, **over})
    env = ShipEnv(cfg, a.assets)
    out = Path(a.out); (out/'checkpoints').mkdir(parents=True, exist_ok=True)
    env.write_xml(out/'scene.xml')
    start = time.monotonic()
    print('Devices:', jax.devices(), flush=True)
    print('Configuration:', asdict(cfg), flush=True)
    (out/'config.json').write_text(json.dumps(asdict(cfg), indent=2))
    (out/'versions.json').write_text(json.dumps({name: importlib.metadata.version(name)
        for name in ['jax', 'jaxlib', 'mujoco', 'mujoco-mjx', 'brax',
                     'flax', 'optax', 'orbax-checkpoint', 'numpy', 'scipy']}, indent=2))

    def save(step, make_policy, params, name=None):
        payload = {'params': jax.device_get(params), 'config': asdict(cfg), 'steps': int(step),
                   'seed': a.seed, 'network': [128, 128, 128], 'task': TASK}
        dest = out/'checkpoints'/(name or f'step_{int(step):09d}.pkl')
        tmp = dest.with_suffix('.tmp')
        with tmp.open('wb') as f:
            pickle.dump(payload, f)
        tmp.replace(dest)
        print(f'checkpoint saved: {dest.name}', flush=True)

    def progress(step, metrics):
        row = {'step': int(step), 'elapsed_s': time.monotonic()-start}
        row.update({k: float(v) for k, v in metrics.items()})
        with (out/'progress.jsonl').open('a') as f:
            f.write(json.dumps(row)+'\n')
        print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()
                          if k in ('step', 'elapsed_s') or k.startswith('episode/')}), flush=True)

    # Brax's built-in evaluator is disabled: on CPU its compilation needs >13 GB, and the
    # policies are judged by evaluate.py on validation waves instead. Training-episode
    # statistics (episode/...) are logged every 200k steps.
    old = None
    if a.restore:
        previous = load(a.restore)
        if previous.get('task') not in (TASK, 'constant-speed-multispeed-v1'):
            raise ValueError('Restore only from this task or the original multispeed notebook (same observation and network)')
        old = previous['params']
        print('Restored parameters from', a.restore, 'task', previous.get('task'), flush=True)
    _, params, _ = ppo.train(
        environment=env, num_timesteps=8 if a.tiny else a.steps,
        num_envs=2 if a.tiny else a.envs, episode_length=10 if a.tiny else a.episode_length,
        action_repeat=1, learning_rate=a.lr, entropy_cost=.01,
        discounting=.99, unroll_length=2 if a.tiny else 16,
        batch_size=2 if a.tiny else 64, num_minibatches=1 if a.tiny else 4,
        num_updates_per_batch=1 if a.tiny else 4,
        normalize_observations=True, network_factory=NETWORK, wrap_env_fn=training_wrap,
        seed=a.seed, num_evals=2 if a.tiny else a.checkpoints + 1,
        num_eval_envs=2 if a.tiny else 8, deterministic_eval=True,
        progress_fn=progress, policy_params_fn=save, run_evals=False,
        restore_params=old, restore_value_fn=True,
        log_training_metrics=True, training_metrics_steps=8 if a.tiny else 200_000,
    )
    save(0, None, params, name='final.pkl')
    (out/'training_time.json').write_text(json.dumps({'steps': a.steps, 'minutes': (time.monotonic()-start)/60}))
    print('Finished. Training reward alone does not establish successful walking.', flush=True)


if __name__ == '__main__':
    main()

### `evaluate.py`
★ 60秒・一定速度・途中リセットなしの評価。全条件を並列に計算します。速度で切り替えるモデルにも対応。1条件ごとの手順と判定は最初のノートブックと同一です。

In [ ]:
%%writefile $WORK_STR/evaluate.py
"""Constant-speed 60 s evaluation, all runs batched (vmap), no episode resets.

Per run this is the same procedure as the original evaluate_speeds.py: reset with
PRNGKey(seed), start x=-12 m, the speed command fixed for the whole run, the same
termination tests and summary metrics. A run that fails is ignored after its failure step.
"""
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')
import argparse,json,time
from dataclasses import asdict
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jp
from brax.training.acme import running_statistics
from brax.training.agents.ppo import networks
from ship_env import ShipEnv,Config,motion
from train import NETWORK,load,TASK

REASONS=['completed','numerical_error','fall_criterion','out_of_bounds']

def summarize(r,reason,speed,seed,start_x,seconds,dt,cfg,policy):
    """r columns: time,x,y,z,vx,vy,vz,slip,roll,pitch,heave,deck_roll,deck_pitch,tgt_h,tgt_r,tgt_p."""
    active=r[r[:,0]>=5.];use=active if len(active) else r
    s={'seed':seed,'speed_m_s':speed,'requested_seconds':seconds,'simulated_seconds':float(r[-1,0]),
       'failed':reason!='completed','termination_reason':reason,'fall_detected':reason=='fall_criterion',
       'completed':reason=='completed','success':reason=='completed',
       'forward_displacement_m':float(r[-1,1]-start_x),'start_x_m':start_x,
       'mean_forward_speed_m_s':float(np.mean(use[:,4])),
       'speed_rmse_m_s':float(np.sqrt(np.mean((use[:,4]-speed)**2))),
       'mean_slip_proxy_m2_s2':float(np.mean(use[:,7])),
       'body_roll_rms_deg':float(np.rad2deg(np.sqrt(np.mean(use[:,8]**2)))),
       'body_pitch_rms_deg':float(np.rad2deg(np.sqrt(np.mean(use[:,9]**2)))),
       'body_roll_std_deg':float(np.rad2deg(np.std(use[:,8]))),
       'body_pitch_std_deg':float(np.rad2deg(np.std(use[:,9]))),
       'deck_tracking_rmse':np.sqrt(np.mean((r[:,10:13]-r[:,13:16])**2,axis=0)).tolist(),
       'metric_window':'t >= 5 s; before failure only; fallback whole trace if duration < 5 s',
       'metric_window_seconds':float(len(use)*dt),'policy':policy,
       'config':{**asdict(cfg),'speed':speed},'no_episode_resets':True}
    if reason!='numerical_error' and not all(np.isfinite(s[k]) for k in ['speed_rmse_m_s','mean_slip_proxy_m2_s2','body_roll_rms_deg']):
        s.update(termination_reason='numerical_error',failed=True,completed=False,success=False,fall_detected=False)
    return s

_CACHE={}
def compiled(assets,cfg):
    """Build the environment and jitted functions once per (assets, config); policy weights are an argument."""
    key=(str(assets),cfg)
    if key in _CACHE:return _CACHE[key]
    env=ShipEnv(cfg,assets)
    def reset(rng,speed):
        state=env.reset(rng);p=state.pipeline_state
        obs=env.baseobs(p.data,p.last,p.previous,speed)
        p=p.replace(speed=speed,history=jp.concatenate([jp.zeros(2030),obs]))
        return state.replace(pipeline_state=p,obs=env.observe(p))
    def measure(state):
        p=state.pipeline_state;d=p.data
        pos,vel=jax.vmap(env.relative)(d);rot=d.xmat[:,env.base_id]
        tgt=jax.vmap(lambda t,w,a:motion(t,w,a)[0])(d.time,p.waves,p.amplitude)
        fin=jp.all(jp.isfinite(d.qpos),axis=1)&jp.all(jp.isfinite(d.qvel),axis=1)&jp.isfinite(state.reward)
        roll=jp.arctan2(rot[:,2,1],rot[:,2,2]);pitch=jp.arcsin(jp.clip(-rot[:,2,0],-1,1))
        rows=jp.concatenate([d.time[:,None],pos,vel,state.metrics['slip_proxy'][:,None],
                             roll[:,None],pitch[:,None],d.qpos[:,:3],tgt],axis=1)
        return rows,fin,rot[:,2,2],d.qpos,d.qvel
    make_policy=networks.make_inference_fn(NETWORK(94,12,preprocess_observations_fn=running_statistics.normalize))
    policy_fn=jax.jit(lambda params,obs,k:make_policy(params,deterministic=True)(obs,k)[0])
    _CACHE[key]=(env,jax.jit(jax.vmap(reset)),jax.jit(jax.vmap(env.step)),jax.jit(measure),policy_fn)
    return _CACHE[key]

def evaluate(assets,out,condition,speeds,seeds,seconds=60.,policy_path=None,allow_cpu=False,save_traj=True,quiet=False,pairs=None):
    if not allow_cpu and jax.default_backend()!='gpu':raise RuntimeError('Select GPU runtime')
    residual=.12;payload=None
    if policy_path:
        payload=load(policy_path)
        if payload.get('task')!=TASK:raise ValueError('Wrong checkpoint task')
        residual=payload['config']['residual_rad']
    # Evaluation config: only the deck amplitude and the residual bound matter (random_start=False).
    cfg=Config(roll_deg=condition[0],pitch_deg=condition[1],heave_m=condition[2],residual_rad=residual)
    env,vreset,vstep,measure,policy_fn=compiled(assets,cfg)
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    env.write_xml(out/'scene.xml')
    runs=[(float(sp),int(sd)) for sp,sd in pairs] if pairs else [(sp,sd) for sp in speeds for sd in seeds];N=len(runs)
    keys=jp.stack([jax.random.PRNGKey(sd) for _,sd in runs]);spd=jp.array([sp for sp,_ in runs],jp.float32)
    inference=(lambda obs,i:policy_fn(payload['params'],obs,jax.random.PRNGKey(i))) if payload else None
    if payload and 'schedule' in payload:
        # Speed-scheduled controller: commands below the switch speed use the low-speed network,
        # the others use the general network. The command is fixed per run, so this is a per-run choice.
        low=(spd<payload['schedule']['switch_speed_m_s'])[:,None]
        inference=lambda obs,i:jp.where(low,policy_fn(payload['schedule']['low_params'],obs,jax.random.PRNGKey(i)),
                                        policy_fn(payload['params'],obs,jax.random.PRNGKey(i)))
    T=round(seconds/env.dt);nq,nv=env.mj_model.nq,env.mj_model.nv
    R=np.zeros((T,N,16),np.float32);Q=np.zeros((T,N,nq),np.float32) if save_traj else None
    V=np.zeros((T,N,nv),np.float32) if save_traj else None
    reason=np.zeros(N,np.int8);end=np.full(N,T-1,int)
    if not quiet:print(f'condition={condition} runs={N} steps={T} policy={policy_path} devices={jax.devices()}',flush=True)
    state=vreset(keys,spd);jax.block_until_ready(state.obs)
    start_x=np.asarray(jax.vmap(env.relative)(state.pipeline_state.data)[0][:,0])
    last=time.monotonic()
    for i in range(T):
        a=inference(state.obs,i) if inference else jp.zeros((N,12))
        state=vstep(state,a);rows,fin,zz,q,v=jax.device_get(measure(state))
        R[i]=rows
        if save_traj:Q[i]=q;V[i]=v
        code=np.zeros(N,np.int8)
        code[(np.abs(rows[:,1])>19.5)|(np.abs(rows[:,2])>1.5)]=3
        code[(rows[:,3]<.26)|(zz<.45)]=2
        code[~fin]=1
        new=(reason==0)&(code>0);reason[new]=code[new];end[new]=i
        if not quiet and (i==0 or time.monotonic()-last>20):
            print(f'sim {rows[0,0]:.1f}/{seconds:.0f}s alive {int((reason==0).sum())}/{N}',flush=True);last=time.monotonic()
        if np.all(reason>0):break
    results=[]
    for j,(sp,sd) in enumerate(runs):
        e=int(end[j]) if reason[j] else min(i,T-1);r=R[:e+1,j]
        s=summarize(r,REASONS[reason[j]],sp,sd,float(start_x[j]),seconds,env.dt,cfg,
                    str(policy_path) if policy_path else 'WTW pretrained only')
        results.append(s)
        if save_traj:
            dest=out/f'speed_{sp:.2f}'/f'seed_{sd}';dest.mkdir(parents=True,exist_ok=True)
            (dest/'summary.json').write_text(json.dumps(s,indent=2))
            idx=[k for k in range(e+1) if (k+1)%2==0 or k==e]
            np.savez_compressed(dest/'trajectory.npz',qpos=Q[idx,j],qvel=V[idx,j],time=r[idx,0])
    (out/'results.json').write_text(json.dumps(results,indent=2))
    if not quiet:
        for s in results:print(f"speed {s['speed_m_s']:.2f} seed {s['seed']}: {s['termination_reason']} at {s['simulated_seconds']:.2f}s",flush=True)
        print(f'completed {sum(s["completed"] for s in results)}/{N}',flush=True)
    if any(s['termination_reason']=='numerical_error' for s in results):
        raise RuntimeError('Nonfinite result before failure; stop instead of counting as a fall')
    return results

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--assets',required=True);p.add_argument('--out',required=True)
    p.add_argument('--condition',nargs=3,type=float,required=True);p.add_argument('--speeds',nargs='+',type=float,default=[.1,.2,.3])
    p.add_argument('--seeds',nargs='+',type=int);p.add_argument('--seconds',type=float,default=60)
    p.add_argument('--pairs-json',help='(speed, seed) pairs from make_validation.py instead of --speeds x --seeds')
    p.add_argument('--policy');p.add_argument('--no-traj',action='store_true');p.add_argument('--allow-cpu',action='store_true');a=p.parse_args()
    if a.seconds<=0 or any(s<=0 or s>.3 for s in a.speeds):p.error('Require positive seconds and speeds <= .30')
    pairs=[(v,s) for v,s,_ in json.loads(Path(a.pairs_json).read_text())['pairs']] if a.pairs_json else None
    if not pairs and not a.seeds:p.error('Give --seeds or --pairs-json')
    evaluate(a.assets,a.out,a.condition,a.speeds,a.seeds,a.seconds,a.policy,a.allow_cpu,not a.no_traj,pairs=pairs)

### `make_validation.py`
★ 検証用の波を、評価用seed以外から自動で選びます（難しいが物理的に歩ける波）。

In [ ]:
%%writefile $WORK_STR/make_validation.py
"""Hard-but-feasible validation conditions (v6).

For each speed, scan wave seeds upward from --first and keep those whose deck-drop index M
(largest downward deck acceleration under the nominal path from x=-12 m, in g) lies in [lo, hi).
Easy waves do not separate good from mediocre policies, and M >= 1 waves cannot be walked at all.
Test seeds 1000-1004 and training ids (>= 10000) are never used.
"""
import os
os.environ.setdefault('JAX_PLATFORMS','cpu')
import argparse,json
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jp
from ship_env import waves_from_id,hardness

def main():
    p=argparse.ArgumentParser();p.add_argument('--out',required=True)
    p.add_argument('--speeds',nargs='+',type=float,default=[.1,.2,.3]);p.add_argument('--per-speed',type=int,default=20)
    p.add_argument('--first',type=int,default=5000);p.add_argument('--last',type=int,default=9999)
    p.add_argument('--lo',type=float,default=.75);p.add_argument('--hi',type=float,default=1.)
    p.add_argument('--window',type=float,default=60.,help='M is measured over the first WINDOW seconds')
    p.add_argument('--max-full',type=float,default=1.,help='and the whole 60 s must stay below this M (feasible)')
    p.add_argument('--condition',nargs=3,type=float,default=[18.,12.,.10]);a=p.parse_args()
    amp=jp.array([a.condition[2],np.deg2rad(a.condition[0]),np.deg2rad(a.condition[1])])
    m_of=jax.jit(jax.vmap(lambda i,v:hardness(waves_from_id(i),amp,v,-12.,seconds=a.window),in_axes=(0,None)))
    full_of=jax.jit(jax.vmap(lambda i,v:hardness(waves_from_id(i),amp,v,-12.),in_axes=(0,None)))
    seeds=np.arange(a.first,a.last+1);seeds=seeds[(seeds<1000)|(seeds>1004)]
    pairs=[]
    for v in a.speeds:
        m=np.asarray(m_of(jp.asarray(seeds),v));mf=np.asarray(full_of(jp.asarray(seeds),v))
        ok=seeds[(m>=a.lo)&(m<a.hi)&(mf<a.max_full)][:a.per_speed]
        if len(ok)<a.per_speed:raise RuntimeError(f'only {len(ok)} waves in range for speed {v}')
        pairs+=[[float(v),int(s),float(m[seeds==s][0])] for s in ok]
        print(f'speed {v:.2f}: seeds {ok[0]}..{ok[-1]} (M {a.lo}-{a.hi}), {len(ok)} waves',flush=True)
    Path(a.out).write_text(json.dumps({'pairs':pairs,'rule':f'M over the first {a.window:g} s in [{a.lo},{a.hi}), whole-run M < {a.max_full}, from seed {a.first} upward','condition':a.condition},indent=1))

if __name__=='__main__':main()

### `select_checkpoint.py`
★ 検証用の波だけで候補を比べ、1つを選びます。評価用seedを渡すとエラーで止まります。

In [ ]:
%%writefile $WORK_STR/select_checkpoint.py
"""Choose ONE checkpoint using validation waves only (test seeds 1000-1004 are never used here).

Rule (fixed before looking at test results):
 1. eligible only if it walks 60 s on a static deck at every speed with speed error RMSE <= --max-static-rmse
    (normal walking and speed tracking must not be broken);
 2. most completed 60 s runs on the hard validation conditions;
 3. ties -> lower speed-tracking RMSE on the completed validation runs; then the later checkpoint.
"""
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')
import argparse,json
from pathlib import Path
import numpy as np
from evaluate import evaluate

def main():
    p=argparse.ArgumentParser();p.add_argument('--assets',required=True);p.add_argument('--run',required=True)
    p.add_argument('--condition',nargs=3,type=float,default=[18.,12.,.10]);p.add_argument('--speeds',nargs='+',type=float,default=[.1,.2,.3])
    p.add_argument('--pairs-json',required=True,help='output of make_validation.py');p.add_argument('--seconds',type=float,default=60)
    p.add_argument('--max-static-rmse',type=float,default=.03);p.add_argument('--include-restore',action='store_true',
                   help='also consider the starting model (step 0) when training was restored from a checkpoint')
    p.add_argument('--allow-cpu',action='store_true');a=p.parse_args()
    pairs=[(v,s) for v,s,_ in json.loads(Path(a.pairs_json).read_text())['pairs']]
    if {s for _,s in pairs}&set(range(1000,1005)):p.error('Validation must not use the test seeds 1000-1004')
    # One results cache per validation set, so changing the set never reuses stale numbers.
    run=Path(a.run);dest=run/('validation_'+Path(a.pairs_json).stem);dest.mkdir(parents=True,exist_ok=True)
    table_path=dest/'validation.json';table=json.loads(table_path.read_text()) if table_path.exists() else {}
    cands=sorted((run/'checkpoints').glob('step_*.pkl'))
    cands=[c for c in cands if int(c.stem.split('_')[1])>0 or a.include_restore]+[None]   # None = public policy only (reference)
    for c in cands:
        name=c.name if c else 'baseline'
        if name in table:continue
        res=evaluate(a.assets,dest/'tmp',a.condition,None,None,a.seconds,str(c) if c else None,a.allow_cpu,save_traj=False,quiet=True,pairs=pairs)
        # On a static deck the seed does not matter (no waves), so one seed per speed suffices.
        flat=evaluate(a.assets,dest/'tmp_flat',[0.,0.,0.],a.speeds,[pairs[0][1]],a.seconds,str(c) if c else None,a.allow_cpu,save_traj=False,quiet=True)
        ok=[r for r in res if r['completed']]
        table[name]={'completed':int(len(ok)),'count':len(res),
                     'mean_survival_s':float(np.mean([r['simulated_seconds'] if r['failed'] else a.seconds for r in res])),
                     'speed_rmse_completed':float(np.mean([r['speed_rmse_m_s'] for r in ok])) if ok else None,
                     'completed_by_speed':{f'{v:.2f}':int(sum(r['completed'] for r in res if r['speed_m_s']==v)) for v in a.speeds},
                     'static_deck_ok':all(r['completed'] for r in flat),
                     'static_rmse':[float(r['speed_rmse_m_s']) for r in flat],
                     'step':int(c.stem.split('_')[1]) if c else 0}
        table_path.write_text(json.dumps(table,indent=2))
        t=table[name]
        print(f"{name}: hard validation {t['completed']}/{len(res)} by speed {t['completed_by_speed']}  rmse {t['speed_rmse_completed']}  "
              f"static ok {t['static_deck_ok']} static rmse {[round(x,3) for x in t['static_rmse']]}",flush=True)
    elig={k:v for k,v in table.items() if k!='baseline' and v['static_deck_ok'] and max(v['static_rmse'])<=a.max_static_rmse}
    if not elig:raise RuntimeError('No checkpoint walks 60 s on a static deck with good speed tracking; do not use this run')
    best=max(elig,key=lambda k:(elig[k]['completed'],-(elig[k]['speed_rmse_completed'] or 9.),elig[k]['step']))
    sel={'selected':best,'path':str(run/'checkpoints'/best),**elig[best],'baseline':table.get('baseline'),
         'rule':f'static deck 60 s at every speed with RMSE <= {a.max_static_rmse}; then max hard-validation completions; tie -> lower speed RMSE; tie -> later step',
         'pairs_json':str(a.pairs_json),'speeds':a.speeds,'condition':a.condition}
    (run/'selected.json').write_text(json.dumps(sel,indent=2));print('SELECTED',json.dumps(sel),flush=True)

if __name__=='__main__':main()

### `schedule.py`
★ 最初のモデルと低速専用モデルを、指令速度で切り替える1つのモデルファイルにまとめます。

In [ ]:
%%writefile $WORK_STR/schedule.py
"""Combine the general policy and a low-speed specialist into one speed-scheduled policy file.

Commands below --switch use the specialist; all other commands use the general policy unchanged.
Both inputs are trusted local pickles (the original notebook's policy and this run's checkpoint).
"""
import argparse,pickle
from train import TASK

def main():
    p=argparse.ArgumentParser();p.add_argument('--general',required=True);p.add_argument('--low',required=True)
    p.add_argument('--switch',type=float,default=.15);p.add_argument('--out',required=True);a=p.parse_args()
    with open(a.general,'rb') as f:gen=pickle.load(f)
    with open(a.low,'rb') as f:low=pickle.load(f)
    out={'task':TASK,'params':gen['params'],'config':{**(low.get('config') or {}),'residual_rad':.12},
         'schedule':{'switch_speed_m_s':a.switch,'low_params':low['params'],'general':str(a.general),'low':str(a.low)}}
    with open(a.out,'wb') as f:pickle.dump(out,f)
    print('wrote',a.out,'switch',a.switch)

if __name__=='__main__':main()

### `report.py`
★ 評価結果の集計表（完走数・転倒数・条件ごとの1対1比較）。

In [ ]:
%%writefile $WORK_STR/report.py
"""Tables for the test seeds: completion counts first, then per-condition outcomes."""
import csv,json,statistics
from pathlib import Path
METRICS=['mean_forward_speed_m_s','speed_rmse_m_s','mean_slip_proxy_m2_s2','body_roll_rms_deg','body_pitch_rms_deg']

def write(path,rows):
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)

def survival(r):return r['simulated_seconds'] if r['failed'] else r['requested_seconds']

def summarize(baseline,trained,out,meta):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    key=lambda r:(r['speed_m_s'],r['seed'])
    bb={key(r):r for r in baseline};tt={key(r):r for r in trained}
    assert sorted(bb)==sorted(tt),'baseline and trained runs differ'
    by_speed=[]
    for sp in sorted({k[0] for k in bb}):
        for label,rows in [('baseline',bb),('trained',tt)]:
            runs=[r for k,r in rows.items() if k[0]==sp];ok=[r for r in runs if r['completed']]
            row={'speed_m_s':sp,'controller':label,'count':len(runs),'completed_count':len(ok),
                 'fall_count':sum(r['fall_detected'] for r in runs),
                 'out_of_bounds_count':sum(r['termination_reason']=='out_of_bounds' for r in runs),
                 'mean_survival_s':statistics.mean(survival(r) for r in runs)}
            for m in METRICS:row[m]=statistics.mean(r[m] for r in ok) if ok else None
            by_speed.append(row)
    paired=[]
    for k in sorted(bb):
        b,t=bb[k],tt[k]
        assert b['config']['speed']==t['config']['speed'] and b['requested_seconds']==t['requested_seconds']
        change=('both_completed' if b['completed'] and t['completed'] else 'improved' if t['completed'] else
                'worse' if b['completed'] else 'both_failed')
        paired.append({'speed_m_s':k[0],'seed':k[1],'baseline':b['termination_reason'],'baseline_survival_s':round(survival(b),2),
                       'trained':t['termination_reason'],'trained_survival_s':round(survival(t),2),'change':change})
    total={label:sum(r['completed'] for r in rows.values()) for label,rows in [('baseline',bb),('trained',tt)]}
    rep={'meta':meta,'completed':total,'count':len(bb),'by_speed':by_speed,'paired':paired,
         'changes':{c:sum(p['change']==c for p in paired) for c in ['improved','worse','both_completed','both_failed']},
         'aggregation':'completion/fall counts use all runs; continuous means use completed runs only'}
    write(out/'speed_summary.csv',by_speed);write(out/'paired_outcomes.csv',paired)
    (out/'final_report.json').write_text(json.dumps(rep,indent=2))
    print(f"completed: baseline {total['baseline']}/{len(bb)}, trained {total['trained']}/{len(bb)}; changes {rep['changes']}",flush=True)
    return rep

### `render_vertical.py`
★ 比較動画。上が学習前（公開方策のみ）、下がv6。保存した軌跡を再生します。

In [ ]:
%%writefile $WORK_STR/render_vertical.py
"""Before/after comparison video, stacked vertically: top = public policy only, bottom = with the learned residual.

Replays saved evaluation trajectories (no physics is re-simulated). Each case is shown for the full
60 s at 1x speed. After a fall the last pose is frozen and labelled. Cases are those test runs where the
public policy falls and the trained policy completes (chosen from the results, as stated in the video).
"""
import os
os.environ.setdefault('MUJOCO_GL','glfw' if os.name=='nt' else 'egl')
os.environ['IMAGEIO_FFMPEG_NO_PREVENT_SIGINT']='1'
import argparse,json
from pathlib import Path
import numpy as np
import mujoco
import imageio.v2 as imageio
from PIL import Image,ImageDraw,ImageFont

FPS=25;W,H=960,540
CJK=['/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc',
     '/mnt/c/Windows/Fonts/meiryo.ttc','C:/Windows/Fonts/meiryo.ttc','/mnt/c/Windows/Fonts/YuGothM.ttc','C:/Windows/Fonts/YuGothM.ttc']
LATIN=['/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf','C:/Windows/Fonts/arial.ttf']
JA={'top':'学習前：公開歩行方策（WTW）のみ','bottom':'学習後：WTW ＋ 学習した関節補正',
    'walk':'歩行中','fall':'転倒','done':'60秒 完走','case':'評価seed {seed} ｜ 指令速度 {speed:.2f} m/s ｜ 甲板の揺れ：ロール18° ピッチ12° 上下0.10 m',
    'title':['揺れる甲板上の四脚ロボット Unitree Go1','上：学習前（公開方策のみ） ／ 下：学習後（補正モデル追加）','学習に使っていない評価用の波（seed 1000〜1004）で比較','等倍速（1倍速）'],
    'case_title':'ケース {k}：評価seed {seed}、指令速度 {speed:.2f} m/s','case_sub':'学習前は {t:.1f} 秒で転倒、学習後は60秒歩行を完了',
    'end':['評価結果（seed 1000〜1004 × 速度0.10/0.20/0.30 m/s ＝ 15条件、各60秒）','学習前：{b}/15 完走 → 学習後：{t}/15 完走','動画のケースは「学習前は転倒・学習後は完走」の条件から選んでいます','MuJoCoシミュレーションの結果です（実機での転倒防止は未検証）']}
EN={'top':'BEFORE: public WTW policy only','bottom':'AFTER: WTW + learned joint residual',
    'walk':'walking','fall':'FALL','done':'completed 60 s','case':'test seed {seed} | command {speed:.2f} m/s | deck: roll 18 deg, pitch 12 deg, heave 0.10 m',
    'title':['Unitree Go1 walking on a moving ship deck','top: before training / bottom: after training','held-out test waves (seeds 1000-1004)','real time (1x)'],
    'case_title':'Case {k}: test seed {seed}, command {speed:.2f} m/s','case_sub':'before: falls at {t:.1f} s; after: completes 60 s',
    'end':['Test result (seeds 1000-1004 x 0.10/0.20/0.30 m/s = 15 runs, 60 s each)','before: {b}/15 completed -> after: {t}/15 completed','video cases were chosen among "before falls, after completes"','MuJoCo simulation only; no claim about the real robot']}

def font(size):
    for p in CJK:
        if Path(p).exists():return ImageFont.truetype(p,size),JA
    for p in LATIN:
        if Path(p).exists():return ImageFont.truetype(p,size),EN
    return ImageFont.load_default(),EN

def card(lines,f,size=(W,2*H+4)):
    im=Image.new('RGB',size,(12,18,28));d=ImageDraw.Draw(im);y=size[1]//2-len(lines)*34
    for i,l in enumerate(lines):
        w=d.textlength(l,font=f);d.text(((size[0]-w)/2,y+i*68),l,font=f,fill='white' if i else (255,214,102))
    return np.asarray(im)

def load_case(root,label,seed,speed):
    f=root/f'test_{label}'/f'speed_{speed:.2f}'/f'seed_{seed}'
    return json.loads((f/'summary.json').read_text()),np.load(f/'trajectory.npz')

def main():
    p=argparse.ArgumentParser();p.add_argument('--root',required=True);p.add_argument('--out',required=True)
    p.add_argument('--cases',nargs='*',default=[],help='seed:speed; default = earliest and latest baseline-fall cases rescued by training')
    p.add_argument('--max-cases',type=int,default=2);p.add_argument('--seconds',type=float,default=60.)
    a=p.parse_args();root=Path(a.root);out=Path(a.out);out.parent.mkdir(parents=True,exist_ok=True)
    base={(r['seed'],r['speed_m_s']):r for r in json.loads((root/'test_baseline/results.json').read_text())}
    trained={(r['seed'],r['speed_m_s']):r for r in json.loads((root/'test_trained/results.json').read_text())}
    rescued=sorted([k for k in base if base[k]['failed'] and trained[k]['completed']],key=lambda k:base[k]['simulated_seconds'])
    if a.cases:cases=[(int(s),float(v)) for s,v in (c.split(':') for c in a.cases)]
    elif len(rescued)<=a.max_cases:cases=rescued
    else:cases=[rescued[0],rescued[-1]][:a.max_cases]
    if not cases:
        msg='No test run where the baseline falls and the trained policy completes; no video made.'
        (out.parent/'video_details.json').write_text(json.dumps({'cases':[],'note':msg},indent=2));print(msg,flush=True);return
    big,T=font(30);small,_=font(22)
    m=mujoco.MjModel.from_xml_path(str(root/'test_baseline/scene.xml'))
    m.vis.global_.offwidth=W;m.vis.global_.offheight=H
    renderer=mujoco.Renderer(m,width=W,height=H);data=[mujoco.MjData(m),mujoco.MjData(m)]
    writer=imageio.get_writer(out,fps=FPS,codec='libx264',macro_block_size=2,quality=8)
    nb=sum(r['completed'] for r in base.values());nt=sum(r['completed'] for r in trained.values())
    try:
        for _ in range(4*FPS):writer.append_data(card(T['title'],big))
        for k,(seed,speed) in enumerate(cases,1):
            (sb,tb),(st,tt)=load_case(root,'baseline',seed,speed),load_case(root,'trained',seed,speed)
            for _ in range(3*FPS):writer.append_data(card([T['case_title'].format(k=k,seed=seed,speed=speed),
                                                           T['case_sub'].format(t=sb['simulated_seconds'])],big))
            cams=[mujoco.MjvCamera(),mujoco.MjvCamera()]
            for c in cams:c.azimuth=90;c.elevation=-18;c.distance=3.6
            for i in range(round(a.seconds*FPS)):
                t=(i+1)/FPS;panels=[]
                for j,(s,tr,label) in enumerate([(sb,tb,T['top']),(st,tt,T['bottom'])]):
                    idx=max(0,min(np.searchsorted(tr['time'],t,side='right')-1,len(tr['time'])-1))
                    data[j].qpos[:]=tr['qpos'][idx];data[j].qvel[:]=tr['qvel'][idx];mujoco.mj_forward(m,data[j])
                    trunk=data[j].body('trunk').xpos;deck=data[j].body('deck').xpos
                    cams[j].lookat[:]=[trunk[0],0.,deck[2]+.35]
                    renderer.update_scene(data[j],camera=cams[j]);im=Image.fromarray(renderer.render());d=ImageDraw.Draw(im)
                    fell=s['failed'] and t>s['simulated_seconds']
                    if fell:status,col=f"{T['fall']} ({s['simulated_seconds']:.1f} s)",(255,90,90)
                    elif t>=a.seconds-.05:status,col=T['done'],(110,230,120)
                    else:status,col=f"{T['walk']}  t = {t:5.1f} s",(255,255,255)
                    d.rectangle((0,0,W,48),fill=(15,20,30));d.text((12,8),label,font=big,fill=(255,214,102) if j else (170,200,255))
                    d.rectangle((0,H-40,W,H),fill=(15,20,30));d.text((12,H-34),T['case'].format(seed=seed,speed=speed),font=small,fill='white')
                    tw=d.textlength(status,font=big);d.rectangle((W-tw-28,56,W-8,100),fill=(15,20,30));d.text((W-tw-18,60),status,font=big,fill=col)
                    panels.append(np.asarray(im))
                frame=np.concatenate([panels[0],np.full((4,W,3),255,np.uint8),panels[1]],axis=0)
                writer.append_data(frame)
            tb.close();tt.close()
        for _ in range(6*FPS):writer.append_data(card([l.format(b=nb,t=nt) for l in T['end']],small))
    finally:
        writer.close();renderer.close()
    (out.parent/'video_details.json').write_text(json.dumps({'cases':[{'seed':s,'speed':v,'baseline_fall_s':base[(s,v)]['simulated_seconds']} for s,v in cases],
        'rescued_cases':[list(k) for k in rescued],'selection':'cases where the baseline falls and the trained policy completes; earliest and latest baseline fall',
        'layout':'top = public policy only, bottom = with learned residual','playback':'1x','after_fall':'final pose frozen and labelled'},indent=2))
    print('video:',out,flush=True)

if __name__=='__main__':main()

### `run_experiment.py`
★ 全体の手順（検証の波→低速専用モデルの学習と選択→評価→静止甲板の対照→動画）。完了済みの段階はスキップします。

In [ ]:
%%writefile $WORK_STR/run_experiment.py
"""v6 pipeline: speed-scheduled controller = original multispeed policy + low-speed specialist.

original policy (first notebook) -> low-speed validation waves -> specialist fine-tuning (several seeds)
-> choose one specialist on validation waves only (or use a specialist file trained beforehand) -> scheduled policy -> test once on seeds 1000-1004
(public WTW only / original policy / scheduled policy) -> static-deck control -> video.
Finished stages are skipped when their inputs (config, code, policies) are unchanged."""
import argparse,hashlib,json,os,queue,shutil,signal,subprocess,sys,threading,time,pickle
from pathlib import Path
from report import summarize
ROOT=Path(__file__).resolve().parent
TASK='ship-residual-v5'
def digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def run(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    print('RUN:',*args,flush=True)
    p=subprocess.Popen([sys.executable,'-u',*map(str,args)],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=os.name!='nt')
    q=queue.Queue()
    def reader():
        for line in p.stdout:q.put(line)
        q.put(None)
    threading.Thread(target=reader,daemon=True).start();start=time.monotonic()
    try:
        with log.open('a',encoding='utf-8') as f:
            while True:
                try:line=q.get(timeout=30)
                except queue.Empty:line=f'[alive] elapsed {(time.monotonic()-start)/60:.1f} min\n'
                if line is None:break
                if 'warp' in line.lower():continue
                print(line,end='',flush=True);f.write(line);f.flush()
        if p.wait():raise RuntimeError(f'Child failed. See {log}')
    finally:
        if p.poll() is None:
            if os.name=='nt':p.terminate()
            else:os.killpg(p.pid,signal.SIGTERM)
            try:p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                if os.name=='nt':p.kill()
                else:os.killpg(p.pid,signal.SIGKILL)
                p.wait()
def stage(dest,expected,fn):
    """Run fn() unless dest/completed.json records the same inputs."""
    marker=Path(dest)/'completed.json'
    if marker.exists() and json.loads(marker.read_text())==expected:print('SKIP completed:',dest,flush=True);return
    fn();Path(dest).mkdir(parents=True,exist_ok=True);marker.write_text(json.dumps(expected,indent=2))
def convert_original(src,dst):
    """The first notebook's policy has the same observation, network and residual bound (0.12 rad); only the task label differs."""
    with open(src,'rb') as f:p=pickle.load(f)   # trusted file: the user's own result of the first notebook
    if p.get('task') not in ('constant-speed-multispeed-v1',TASK):raise ValueError('Not a policy of the first notebook')
    p=dict(p,task=TASK,config={**(p.get('config') or {}),'residual_rad':.12},source_task=p.get('task'))
    Path(dst).parent.mkdir(parents=True,exist_ok=True)
    with open(dst,'wb') as f:pickle.dump(p,f)
def main():
    parser=argparse.ArgumentParser();parser.add_argument('--config',required=True);a=parser.parse_args()
    cfg=json.loads(Path(a.config).read_text());out=Path(cfg['output']);out.mkdir(parents=True,exist_ok=True)
    saved=out/'experiment_config.json'
    if saved.exists() and json.loads(saved.read_text())!=cfg:raise RuntimeError('Config changed; use a new OUTPUT')
    saved.write_text(json.dumps(cfg,indent=2))
    cpu=['--allow-cpu'] if cfg.get('allow_cpu') else []
    sea=cfg['test_condition'];sp=cfg['specialist'];low=sp['validation_speed']
    core=['model.py','policy.py','ship_env.py','reset_wrapper.py','train.py']
    # Only files that change the numbers enter the signature (editing the video script does not retrain).
    signature={'config':{k:v for k,v in cfg.items() if k!='video'},
               'source':{n:digest(ROOT/n) for n in core+['evaluate.py','select_checkpoint.py','make_validation.py','schedule.py']},
               'public_policy':digest(Path(cfg['assets'])/'policy.npz'),'original_policy':digest(cfg['original_policy']),
               'specialist_policy':digest(cfg['specialist_policy']) if cfg.get('specialist_policy') else None}
    # 0) the first notebook's policy, used unchanged for commands >= switch speed
    original=out/'original/original_policy.pkl'
    stage(out/'original',{'signature':signature,'stage':'original'},lambda:convert_original(cfg['original_policy'],original))
    # 1) low-speed validation waves (never the test seeds): whole-run hard waves + waves that are hard in the first 10 s
    val=out/'validation'
    def make_val():
        v=sp['validation']
        run(['make_validation.py','--out',val/'hard.json','--speeds',low,'--per-speed',v['hard_count'],'--first',v['hard_first'],
             '--lo',v['hard_lo'],'--hi',v['hard_hi'],'--condition',*sea],val/'console.log')
        run(['make_validation.py','--out',val/'early.json','--speeds',low,'--per-speed',v['early_count'],'--first',v['early_first'],
             '--window',v['early_window_s'],'--lo',v['early_lo'],'--hi',v['early_hi'],'--condition',*sea],val/'console.log')
        d=[json.loads((val/f).read_text()) for f in ('hard.json','early.json')]
        pairs=[p for x in d for p in x['pairs']];assert len({(p[0],p[1]) for p in pairs})==len(pairs)
        (val/'low_speed_pairs.json').write_text(json.dumps({'pairs':pairs,'rule':' + '.join(x['rule'] for x in d),'condition':sea},indent=1))
    stage(val,{'signature':signature,'stage':'validation'},make_val)
    pairs=val/'low_speed_pairs.json'
    if cfg.get('specialist_policy'):
        # a low-speed specialist trained beforehand with this same procedure (no training here)
        low_path=Path(cfg['specialist_policy']);print('Using the given low-speed specialist (no training):',low_path,flush=True)
        chosen={'source':'given file','path':str(low_path),'sha256':digest(low_path)}
    else:
        # 2) low-speed specialist: fine-tune the original policy on commands 0.10-0.15 m/s, several seeds;
        #    each run keeps its best checkpoint on the validation waves (the original itself is a candidate)
        picks=[]
        for s in sp['seeds']:
            run_dir=out/f'specialist_seed{s}'
            stage(run_dir,{'signature':signature,'stage':'train','seed':s},lambda:run(['train.py','--assets',cfg['assets'],'--out',run_dir,
                  '--restore',original,'--steps',sp['steps'],'--envs',sp['envs'],'--seed',s,'--checkpoints',sp['checkpoints'],
                  '--episode-length',sp['episode_length'],'--lr',sp['lr'],'--config-json',json.dumps(sp['env_overrides']),*cpu],run_dir/'console.log'))
            stage(run_dir/f'validation_{pairs.stem}',{'signature':signature,'stage':'select','seed':s},lambda:run(['select_checkpoint.py',
                  '--assets',cfg['assets'],'--run',run_dir,'--condition',*sea,'--speeds',low,'--pairs-json',pairs,
                  '--include-restore',*cpu],run_dir/'select_console.log'))
            picks.append({**json.loads((run_dir/'selected.json').read_text()),'seed':s})
        # 3) one specialist over all seeds, same rule: most validation completions, then lower speed error, then later step
        best=max(picks,key=lambda r:(r['completed'],-(r['speed_rmse_completed'] or 9.),r['step']))
        low_path=Path(best['path']);chosen={'source':'trained here','selected':best,'candidates':picks}
        print('SPECIALIST',json.dumps({k:best[k] for k in ('seed','selected','completed','count','speed_rmse_completed')}),flush=True)
    (out/'specialist_selected.json').write_text(json.dumps(chosen,indent=2))
    policy=out/'scheduled_policy.pkl'
    stage(out/'schedule',{'signature':signature,'stage':'schedule','low':digest(low_path)},lambda:run(['schedule.py',
          '--general',original,'--low',low_path,'--switch',cfg['switch_speed_m_s'],'--out',policy],out/'schedule_console.log'))
    # validation waves: the original policy vs the low-speed specialist (both at the low validation speed)
    comparison={}
    for label,pol in [('original',original),('specialist',low_path)]:
        dest=out/f'validation_{label}'
        stage(dest,{'signature':signature,'policy':digest(pol)},lambda:run(['evaluate.py','--assets',cfg['assets'],'--out',dest,
              '--condition',*sea,'--pairs-json',pairs,'--seconds',cfg['seconds'],'--no-traj','--policy',pol,*cpu],dest/'console.log'))
        r=json.loads((dest/'results.json').read_text());ok=[x for x in r if x['completed']]
        comparison[label]={'completed':len(ok),'count':len(r),
                           'speed_rmse_completed':sum(x['speed_rmse_m_s'] for x in ok)/len(ok) if ok else None}
    (out/'validation_comparison.json').write_text(json.dumps(comparison,indent=2))
    print('VALIDATION',json.dumps(comparison),flush=True)
    # 4) test once: public WTW only / original policy / scheduled policy; same waves and speeds, 60 s, no resets.
    # The command is fixed per run, so the scheduled controller's runs at commands >= switch are exactly the original
    # policy's runs: they are taken from the original policy's evaluation, not simulated a second time (on GPU a second
    # simulation of the same controller can differ slightly and flip a marginal run). The specialist is evaluated on
    # the commands below the switch, and the two parts are joined into {prefix}_trained.
    switch=cfg['switch_speed_m_s'];low_speeds=[v for v in cfg['speeds'] if v<switch]
    def evaluate_all(prefix,condition,traj):
        res={}
        for label,pol,speeds in [('baseline',None,cfg['speeds']),('original',original,cfg['speeds']),('trained_low',low_path,low_speeds)]:
            if not speeds:continue
            dest=out/f'{prefix}_{label}'
            args=['evaluate.py','--assets',cfg['assets'],'--out',dest,'--condition',*condition,'--speeds',*speeds,
                  '--seeds',*cfg['test_seeds'],'--seconds',cfg['seconds'],*cpu]+(['--policy',pol] if pol else [])+([] if traj else ['--no-traj'])
            stage(dest,{'signature':signature,'policy':digest(pol) if pol else None},lambda:run(args,dest/'console.log'))
            res[label]=json.loads((dest/'results.json').read_text())
        low={(r['speed_m_s'],r['seed']):r for r in res.get('trained_low',[])};joined=[]
        dest=out/f'{prefix}_trained'
        if dest.exists():shutil.rmtree(dest)
        dest.mkdir(parents=True)
        for r in res['original']:
            src,part=(out/f'{prefix}_trained_low',low[(r['speed_m_s'],r['seed'])]) if r['speed_m_s']<switch else (out/f'{prefix}_original',r)
            joined.append({**part,'policy':str(policy),'network':'low-speed specialist' if r['speed_m_s']<switch else 'original'})
            run_dir=Path('speed_%.2f'%r['speed_m_s'])/f"seed_{r['seed']}"
            if traj:shutil.copytree(src/run_dir,dest/run_dir)
        (dest/'results.json').write_text(json.dumps(joined,indent=2))
        (dest/'joined.json').write_text(json.dumps({'switch_speed_m_s':switch,'below_switch':str(low_path),'at_or_above_switch':str(original),
            'note':'runs at commands >= switch are the original policy runs of the same evaluation'},indent=2))
        print(f"{prefix}: scheduled controller = specialist runs below {switch} m/s + original policy runs at or above",flush=True)
        res['trained']=joined
        return res
    meta={'condition':sea,'seeds':cfg['test_seeds'],'speeds':cfg['speeds'],'seconds':cfg['seconds'],'policy':str(policy),'policy_sha256':digest(policy)}
    test=evaluate_all('test',sea,True)
    summarize(test['baseline'],test['trained'],out/'report',meta)
    summarize(test['original'],test['trained'],out/'report_vs_original',meta)
    # 5) static deck control (does the controller still walk normally?)
    if cfg.get('flat_control',True):
        flat=evaluate_all('flat',[0,0,0],False)
        summarize(flat['baseline'],flat['trained'],out/'report_flat',{'condition':[0,0,0]})
        summarize(flat['original'],flat['trained'],out/'report_flat_vs_original',{'condition':[0,0,0]})
    # 6) vertical before/after video (top: public WTW only, bottom: scheduled controller)
    if cfg.get('video',True):
        run(['render_vertical.py','--root',out,'--out',out/'videos/before_after_vertical.mp4','--seconds',cfg['seconds']],out/'render_console.log')
    print('All done:',out,flush=True)
if __name__=='__main__':
    if os.name!='nt':signal.signal(signal.SIGTERM,lambda *_:(_ for _ in ()).throw(KeyboardInterrupt()))
    main()

### `fetch_assets.py`
公開資産（WTWの重み、MuJoCo MenagerieのGo1モデル）をコミット固定で取得し、SHA-256で検証。最初のノートブックと同一。

In [ ]:
%%writefile $WORK_STR/fetch_assets.py
"""Download pinned public files; validate every SHA-256 before conversion."""
import argparse,hashlib,json,urllib.request
from pathlib import Path
def main():
    p=argparse.ArgumentParser();p.add_argument('--assets',default='assets');a=p.parse_args()
    root=Path(a.assets);root.mkdir(parents=True,exist_ok=True)
    manifest=json.loads(Path(__file__).with_name('assets_manifest.json').read_text())
    for row in manifest['files']:
        dest=root/row['path'];dest.parent.mkdir(parents=True,exist_ok=True)
        if not dest.exists() or hashlib.sha256(dest.read_bytes()).hexdigest()!=row['sha256']:
            print('Download:',row['path'],flush=True)
            request=urllib.request.Request(row['url'],headers={'User-Agent':'Go1-ship-notebook/1.0'})
            data=urllib.request.urlopen(request,timeout=120).read()
            if hashlib.sha256(data).hexdigest()!=row['sha256']:raise RuntimeError('SHA256 mismatch: '+row['path'])
            temp=dest.with_suffix(dest.suffix+'.tmp');temp.write_bytes(data);temp.replace(dest)
        print('SHA256 OK:',row['path'],flush=True)
    (root/'provenance.json').write_text(json.dumps(manifest,indent=2))
if __name__=='__main__':main()

### `convert.py`
WTWの重みをPyTorchからJAX用に変換し、出力の一致を確認。最初のノートブックと同一。

In [ ]:
%%writefile $WORK_STR/convert.py
"""Convert only pinned upstream weights, and compare PyTorch with JAX."""
import argparse, json, pickle, io
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import jax
import jax.numpy as jp

def main():
    p=argparse.ArgumentParser();p.add_argument('--assets',required=True);a=p.parse_args()
    root=Path(a.assets)
    w=torch.load(root/'ac_weights_last.pt',map_location='cpu',weights_only=True)
    arrays={k:v.numpy() for k,v in w.items() if k.startswith(('actor_body.','adaptation_module.'))}
    assert arrays['actor_body.0.weight'].shape==(512,2102)
    assert arrays['adaptation_module.0.weight'].shape==(256,2100)
    np.savez(root/'policy.npz',**arrays)
    torch.storage._load_from_bytes=lambda b:torch.load(io.BytesIO(b),map_location='cpu',weights_only=False)
    upstream=pickle.load(open(root/'parameters.pkl','rb'));cfg=upstream['Cfg']
    sections=['env','init_state','control','obs_scales','normalization','sim']
    (root/'policy_config.json').write_text(json.dumps({k:cfg[k] for k in sections},default=lambda x:x.tolist() if hasattr(x,'tolist') else x))
    assert cfg['env']['num_observations']==70 and cfg['env']['num_observation_history']==30
    assert upstream['AC_Args']['activation']=='elu'
    assert cfg['commands']['num_commands']==15
    assert cfg['env']['observe_clock_inputs'] and cfg['env']['observe_two_prev_actions']
    assert not any(cfg['env'][k] for k in ['observe_vel','observe_yaw','observe_contact_states','observe_only_lin_vel'])
    assert cfg['control']['stiffness']['joint']==20 and cfg['control']['damping']['joint']==.5
    from model import NAMES,HOME
    np.testing.assert_allclose([cfg['init_state']['default_joint_angles'][n] for n in NAMES],HOME)
    assert cfg['obs_scales']['dof_pos']==1 and cfg['obs_scales']['dof_vel']==.05
    assert cfg['control']['action_scale']==.25 and cfg['control']['hip_scale_reduction']==.5
    assert cfg['sim']['dt']*cfg['control']['decimation']==.02
    from policy import FrozenPolicy
    frozen=FrozenPolicy(root/'policy.npz')
    h=np.random.default_rng(0).normal(size=(16,2100)).astype(np.float32)
    def net(x,prefix,layers):
        for i in layers:
            x=F.linear(x,w[f'{prefix}.{i}.weight'],w[f'{prefix}.{i}.bias'])
            if i!=layers[-1]:x=F.elu(x)
        return x
    with torch.no_grad():
        th=torch.from_numpy(h);lat=net(th,'adaptation_module',[0,2,4])
        expected=net(torch.cat([th,lat],dim=-1),'actor_body',[0,2,4,6]).numpy()
    actual=np.asarray(jax.jit(frozen)(jp.asarray(h)))
    np.testing.assert_allclose(actual,expected,rtol=2e-4,atol=2e-4)
    report={'torch_jax_max_abs_error':float(np.max(np.abs(actual-expected))),
        'observation_size':70,'history':30,'joint_order':'FL FR RL RR, hip thigh calf',
        'control_dt_s':.02,'conversion_passed':True,
        'actuation':'WTW hardware deployment PD 20/0.5; original Isaac training actuator_net is not reproduced'}
    (root/'conversion_report.json').write_text(json.dumps(report,indent=2));print(json.dumps(report),flush=True)
if __name__=='__main__':main()

### `requirements.txt`
ライブラリのバージョン固定。最初のノートブックと同一。

In [ ]:
%%writefile $WORK_STR/requirements.txt
jax[cuda12]==0.6.2
mujoco==3.3.7
mujoco-mjx==3.3.7
brax==0.12.5
flax==0.11.2
optax==0.2.5
chex==0.1.90
toolz==1.0.0
orbax-checkpoint==0.11.23
numpy==2.2.6
scipy==1.15.3
imageio==2.37.0
imageio-ffmpeg==0.6.0
Pillow==11.3.0

### `assets_manifest.json`
公開資産のURLとSHA-256。最初のノートブックと同一。

In [ ]:
%%writefile $WORK_STR/assets_manifest.json
{
  "wtw_commit": "0e7236bdc81ce855cbe3d70345a7899452bdeb1c",
  "menagerie_commit": "0059d4335f8156206f63a35662313385f7ad6d74",
  "files": [
    {
      "path": "ac_weights_last.pt",
      "url": "https://raw.githubusercontent.com/Improbable-AI/walk-these-ways/0e7236bdc81ce855cbe3d70345a7899452bdeb1c/runs/gait-conditioned-agility/pretrain-v0/train/025417.456545/checkpoints/ac_weights_last.pt",
      "sha256": "d1542b0b8c7b767753a286f1cbabe5b4dc86e67611fb95844e870b3c61f8869e"
    },
    {
      "path": "parameters.pkl",
      "url": "https://raw.githubusercontent.com/Improbable-AI/walk-these-ways/0e7236bdc81ce855cbe3d70345a7899452bdeb1c/runs/gait-conditioned-agility/pretrain-v0/train/025417.456545/parameters.pkl",
      "sha256": "eeb132fe125fb4a2169315ae4e3f2d1b1c0a7857fdcb48824d6d375a26268e11"
    },
    {
      "path": "WTW_LICENSE",
      "url": "https://raw.githubusercontent.com/Improbable-AI/walk-these-ways/0e7236bdc81ce855cbe3d70345a7899452bdeb1c/LICENSE",
      "sha256": "7dc66e1919dbcc2b49af464ceac3518b230e0e253d57988d418e23f01ef675b9"
    },
    {
      "path": "unitree_go1/CHANGELOG.md",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/CHANGELOG.md",
      "sha256": "0cf56d81d15241511c79e8c5e83dcd069b64df30103e97227207e4dfbf0775dc"
    },
    {
      "path": "unitree_go1/LICENSE",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/LICENSE",
      "sha256": "2b794acf1e250f5545ae5a47d81e5685ea5488ce077d1087ed8be5f81e9ba5c0"
    },
    {
      "path": "unitree_go1/README.md",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/README.md",
      "sha256": "a4b63613124da20b09f9f8c550482d670ab2b784a272a489f94e2288df43f151"
    },
    {
      "path": "unitree_go1/assets/calf.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/calf.stl",
      "sha256": "ad3685e132454fd64e4ca3e5089f0d812fb53958e95c14913a22e3193e9e9ed8"
    },
    {
      "path": "unitree_go1/assets/hip.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/hip.stl",
      "sha256": "b6599b9dd2751ebd87449203223f5683a6c235268b76fcc443e00e0d974a06c0"
    },
    {
      "path": "unitree_go1/assets/thigh.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/thigh.stl",
      "sha256": "bbf7c0a441323088c59276dbec8a32e708a1c61f42d30a6f6775176366ea23f8"
    },
    {
      "path": "unitree_go1/assets/thigh_mirror.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/thigh_mirror.stl",
      "sha256": "1a90b89de47a57cb12a2d703f6b008a6ace8a1890a381c27723296fd0d00b9d8"
    },
    {
      "path": "unitree_go1/assets/trunk.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/trunk.stl",
      "sha256": "051a57995dc6f892fc9bc53cd94e9a894f3dc209fd646e94d55d764150013970"
    },
    {
      "path": "unitree_go1/go1.xml",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/go1.xml",
      "sha256": "d5a7466784c8e72fd174cc7fa5a92754a1a0a6d458a1dec79e7ca0abc9525c4a"
    },
    {
      "path": "unitree_go1/scene.xml",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/scene.xml",
      "sha256": "eceb4231bf2a3b0a190296963b38b904ed77dc14f5770bbcdd67fb0be8c49374"
    }
  ]
}

## 4 実行環境を用意
Colabでは、標準環境と分けた専用の仮想環境に、固定バージョンのJAX（CUDA 12版）・MuJoCo・Braxなどを入れます（最初のノートブックと同じ方法）。
動画の日本語表示のため、Colabでは日本語フォント（Noto CJK）も入れます。Colab以外では、`requirements.txt`を入れた環境でこのノートブックを開いてください。

In [ ]:
import json, subprocess, hashlib, queue, threading, time, signal
ENV = os.environ.copy()
for name in ['PYTHONPATH', 'PYTHONHOME', 'JAX_PLATFORMS']:
    ENV.pop(name, None)
ENV.update(PYTHONNOUSERSITE='1', MUJOCO_GL='egl', XLA_PYTHON_CLIENT_PREALLOCATE='false', IMAGEIO_FFMPEG_NO_PREVENT_SIGINT='1')
if ALLOW_CPU:
    ENV['JAX_PLATFORMS'] = 'cpu'
if IN_COLAB:
    VENV = Path('/content/unitree_go1_ship_v6_venv'); PY = VENV / 'bin/python'
    if not PY.exists():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True, env=ENV)
    stamp = VENV / 'requirements.sha256'
    signature = hashlib.sha256((WORK / 'requirements.txt').read_bytes() + b'torch-cpu-2.7.1').hexdigest()
    pip = [sys.executable, '-m', 'pip', '--python', str(PY)]
    if not stamp.exists() or stamp.read_text() != signature:
        subprocess.run(pip + ['install', 'torch==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True, env=ENV)
        subprocess.run(pip + ['install', '-r', str(WORK / 'requirements.txt')], check=True, env=ENV)
        subprocess.run(pip + ['check'], check=True, env=ENV)
        stamp.write_text(signature)
    subprocess.run('apt-get -qq install -y fonts-noto-cjk > /dev/null', shell=True)
else:
    PY = Path(sys.executable)
print('Python:', PY)

def run_child(args, log_name, cpu=False):
    child_env = ENV.copy()
    if cpu:
        child_env['JAX_PLATFORMS'] = 'cpu'
    p = subprocess.Popen([str(PY), '-u', *map(str, args)], cwd=WORK, env=child_env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        for line in p.stdout:
            messages.put(line)
        messages.put(None)
    threading.Thread(target=reader, daemon=True).start()
    started = time.monotonic()
    try:
        with (Path(OUTPUT) / log_name).open('a', encoding='utf-8') as log:
            while True:
                try:
                    line = messages.get(timeout=30)
                except queue.Empty:
                    line = f'[alive] 経過 {(time.monotonic() - started) / 60:.1f} 分（計算中であることだけを示します）\n'
                if line is None:
                    break
                if 'warp' in line.lower():
                    continue
                print(line, end='', flush=True); log.write(line); log.flush()
        if p.wait():
            raise RuntimeError(f'処理が停止しました。直前の出力と {log_name} を確認してください。')
    finally:
        if p.poll() is None:
            os.killpg(p.pid, signal.SIGTERM)
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL); p.wait()

## 5 GPUと公開資産の確認
公開資産をコミットとSHA-256を固定して取得し、WTWの重みをJAX用に変換して、PyTorchとの出力一致を確認します（最初のノートブックと同じ検査）。

In [ ]:
if not ALLOW_CPU:
    subprocess.run(['nvidia-smi'], check=True)
    run_child(['-c', "import jax; print(jax.__version__, jax.devices()); assert jax.default_backend()=='gpu'"], 'gpu_check.log')
run_child(['fetch_assets.py', '--assets', CONFIG['assets']], 'fetch.log', cpu=True)
run_child(['convert.py', '--assets', CONFIG['assets']], 'conversion.log', cpu=True)
print(json.loads((Path(CONFIG['assets']) / 'conversion_report.json').read_text()))
(WORK / 'experiment.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')

## 6 検証の波 → 低速専用モデル → 評価 → 対照 → 動画（長時間）
既定（`TRAIN_SPECIALIST = False`）では2・3を行わず、著者が同じ手順で学習・選択した低速専用モデルを使います。
どちらの場合も、検証の30条件で最初のモデルと低速専用モデルを比べた結果を表示します。

1. **検証の波を選ぶ**：0.10 m/sで、甲板の最大下向き加速度（重力加速度との比、指標M）が0.75以上1未満の波を、seed 5000から順に20個選びます（Mが1以上だと足が甲板から離れ、物理的に歩けません）。
   さらに、**最初の10秒間**のMが0.8以上1未満の波（序盤に甲板が急に落ちる波）を、seed 7000から順に10個選びます。評価用のseed 1003以外の転倒は、この「序盤の急落下」で起きていたためです。
2. **低速専用モデルの学習**：最初のモデルを出発点に、指令速度0.10〜0.15 m/sだけでPPOを続けます（200万ステップ、学習率1×10⁻⁴、30秒のエピソード）。主な工夫は次のとおりです。
   - 序盤に急落下する波を多めに出す（`early_bonus`）
   - 胴体の高さが転倒判定（甲板から0.26 m）に5 cm以内まで近づくと罰を与え、着地の余裕を保たせる（`margin_weight`）
   - 学習率を通常の1/3にして、最初のモデルから離れすぎないようにする
3. **選択**：学習の揺らぎに備えて乱数seedを変えて3回学習し、保存した候補（最初のモデル自身も含む）を検証の30条件で比べます。静止甲板を0.10 m/sで60秒歩けて速度誤差が0.03 m/s以下のもののうち、完走数が最も多いものを選びます。
4. **評価**：公開方策のみ・最初のモデル・v6（切り替え）の3つを、評価用の波1000〜1004×3速度で各60秒走らせます（途中リセットなし）。v6の0.20・0.30 m/sは最初のモデルと同じ制御器なので、最初のモデルの走行をそのまま使い、0.10 m/sだけ低速専用モデルで走らせます。
5. **静止甲板の対照**と、**動画**（学習前は転倒しv6で完走した条件を上下に並べる）。

30秒ごとの`[alive]`表示は、処理が動いていることだけを示します。Colabが切断された場合は、上から再実行すると完了済みの段階をスキップします。

In [ ]:
run_child(['run_experiment.py', '--config', WORK / 'experiment.json'], 'experiment_console.log')

## 7 結果の表
**最初に完走数（60秒、転倒・逸脱なし）を見てください。** 速度誤差などの連続値は、完走した試行だけの平均です。
1対1の比較表の`improved`は「比較相手は転倒、v6は完走」、`worse`はその逆です。

In [ ]:
import pandas as pd
from IPython.display import display
R = Path(OUTPUT)
rep = json.loads((R / 'report/final_report.json').read_text())
vs = json.loads((R / 'report_vs_original/final_report.json').read_text())
print(f"評価（seed 1000〜1004 × 3速度、各60秒）: 公開方策のみ {rep['completed']['baseline']}/{rep['count']}、"
      f"最初のモデル {vs['completed']['baseline']}/{vs['count']}、v6 {rep['completed']['trained']}/{rep['count']} 完走")
print('v6と最初のモデルの1対1比較:', vs['changes'])
table = pd.DataFrame({'公開方策のみ': {(p['seed'], p['speed_m_s']): f"{p['baseline']} {p['baseline_survival_s']}" for p in rep['paired']},
                      '最初のモデル': {(p['seed'], p['speed_m_s']): f"{p['baseline']} {p['baseline_survival_s']}" for p in vs['paired']},
                      'v6': {(p['seed'], p['speed_m_s']): f"{p['trained']} {p['trained_survival_s']}" for p in vs['paired']}})
table.index.names = ['seed', 'speed_m_s']
display(table.sort_index(level=[1, 0]))
s = pd.read_csv(R / 'report_vs_original/speed_summary.csv').replace({'baseline': '最初のモデル', 'trained': 'v6'})
print('速度別（連続値は完走した試行のみの平均）:')
display(s.round(3))
if (R / 'report_flat_vs_original/speed_summary.csv').exists():
    print('静止甲板の対照:')
    display(pd.read_csv(R / 'report_flat_vs_original/speed_summary.csv').replace({'baseline': '最初のモデル', 'trained': 'v6'}).round(3))
vc = json.loads((R / 'validation_comparison.json').read_text())
print(f"検証の波（0.10 m/s、評価用seedとは別の{vc['original']['count']}条件）: 最初のモデル {vc['original']['completed']}、"
      f"低速専用モデル {vc['specialist']['completed']} 完走")
sel = json.loads((R / 'specialist_selected.json').read_text())
if sel['source'] == 'trained here':
    print('このノートブックで学習した低速専用モデルの候補（各学習seedで検証成績が最良のもの）:')
    display(pd.DataFrame(sel['candidates'])[['seed', 'selected', 'completed', 'count', 'speed_rmse_completed', 'static_rmse']])
    print('選ばれた低速専用モデル: seed', sel['selected']['seed'], sel['selected']['selected'])
else:
    print('低速専用モデル: 配布ファイルを使用（SHA-256', sel['sha256'][:16] + '…）')

## 8 グラフ
条件ごとの生存時間（転倒した場合は転倒までの秒数、完走は60秒）です。

In [ ]:
import matplotlib.pyplot as plt
pb = pd.read_csv(R / 'report/paired_outcomes.csv'); po = pd.read_csv(R / 'report_vs_original/paired_outcomes.csv')
labels = [f"{s}\n{v:.1f}" for s, v in zip(pb['seed'], pb['speed_m_s'])]
x = range(len(pb)); w = 0.27
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar([i - w for i in x], pb['baseline_survival_s'], w, label='public WTW only', color='#b8c0c8')
ax.bar(list(x), po['baseline_survival_s'], w, label='first notebook model', color='#e0a040')
ax.bar([i + w for i in x], pb['trained_survival_s'], w, label='v6 (speed-scheduled)', color='#2a7ab0')
ax.axhline(60, color='k', lw=.8, ls='--')
ax.set_xticks(list(x)); ax.set_xticklabels(labels, fontsize=8)
ax.set(xlabel='test seed / command speed [m/s]', ylabel='survival time [s]', ylim=(0, 65)); ax.legend(loc='lower right')
fig.tight_layout(); fig.savefig(R / 'survival_by_condition.png', dpi=180); plt.show()

## 9 比較動画
上が学習前（公開方策のみ）、下がv6です。同じ波・同じ速度・同じ開始位置です。転倒した側は最後の姿勢で止め、「転倒」と表示します。
動画の条件は「学習前は転倒し、v6は完走した条件」の中から、転倒が最も早い条件と最も遅い条件を選んでいます（結果を見て選んでいることを動画内にも記載）。
上下で甲板の揺れ方が違って見えるのは、ロボットの位置が違うためです（甲板は中心のまわりに回転するので、中心から遠いほど上下に大きく動きます。転倒して止まった側と歩き続けた側では位置が変わります）。

**動画の保存場所**：Colabでは、Google Driveの「マイドライブ › ship_robot_runs › experiment62_ship_v6 › videos › before_after_vertical.mp4」です（60秒×2条件で約60 MB）。
このセルはブラウザへのダウンロードも試みますが、ブラウザがポップアップやダウンロードを止めることがあるため、見当たらない場合はDriveから開いてください。

In [ ]:
from IPython.display import Video
video = R / 'videos/before_after_vertical.mp4'
print(json.loads((R / 'videos/video_details.json').read_text()))
print('動画ファイル:', video, f'（{video.stat().st_size / 1e6:.0f} MB）' if video.exists() else '（まだありません。出力フォルダーの render_console.log を確認してください）')
if video.exists() and video.stat().st_size < 20e6:
    display(Video(str(video), embed=True, width=480))
elif video.exists():
    print('動画が大きいため、ここでは表示しません。上の場所（Google Drive）から開いてください。')
if IN_COLAB and video.exists():
    from google.colab import files
    files.download(str(video))

## 補足：設計の理由・検証範囲・注意
### なぜ「速度で切り替える」のか（著者の事前検証から）
- **最初のモデルの弱点は0.10 m/sに集中**：評価の15条件で最初のモデルが転倒したのは、seed 1003（3速度）と、0.10 m/sのseed 1000・1001です。seed 1003は約4.8秒で甲板が重力加速度より速く落ちる（足が甲板から離れる）ため、どのモデルでも歩けません。
- **全速度をまとめて追加学習すると、別の条件を失う**：全速度で最初のモデルを追加学習した十数通りの設定（報酬、エピソード長、開始位置、難しい波の選び方、学習率、重みの平均化など）を試しました。0.10 m/sのseed 1001は救えるようになる一方で、最初のモデルが確実に完走していたseed 1002（約4秒で甲板が急に落ち、着地で胴体が沈む条件）を失い、評価では8〜10/15にとどまりました。
- **偶然ではないことの確認**：開始位置を前後±4 cm・左右±2 cmずらした9通りで同じ波を走らせると、最初のモデルのseed 1002（0.10・0.20 m/s）の完走は9/9で、偶然ではありませんでした。追加学習モデルのseed 1002の転倒も9通りすべてで再現しました。
- **そこで、強い速度帯は最初のモデルをそのまま使い、弱い0.10 m/sだけを専用モデルに任せる**設計にしました。0.20・0.30 m/sの結果は最初のモデルと完全に同じになり、0.10 m/sだけが変わります。

### 低速専用モデルの工夫
- **序盤の急落下を多めに学習**（`early_bonus`）：学習のエピソードごとに4つの波の候補を作り、最初の10秒間の指標Mが0.7以上1.05未満の波を選びやすくします。
- **着地の余裕**（`margin_weight`）：胴体の高さが甲板から0.31 mを下回ると罰を与え、0.26 m（転倒判定）で最大にします。追加学習モデルがseed 1002で転倒したのは、着地で胴体が0.246 mまで沈んだためでした（最初のモデルは0.28 m）。
- **学習率を1/3に**：最初のモデルから離れすぎないようにします。
- **選択は検証の波だけで**：評価用seedは学習にも選択にも使っていません。最初のモデル自身も候補に入れているので、追加学習で検証成績が上がらなければ、最初のモデルがそのまま選ばれます。

### 検証範囲（著者が確認したこと・していないこと）
- 確認済み（CPU）：全ファイルの通し実行（縮小設定）、切り替えの仕組み（両方に最初のモデルを入れると最初のモデルと同じ結果になること）、検証の波の選び方が著者の実験と同一になること。
- 学習の再現性：GPUの並列計算は実行ごとに加算順序が変わるため、同じ設定でも学習結果は完全には一致しません。このため低速専用モデルを3つの乱数seedで学習し、検証で選んでいます。著者の3回の学習では評価の完走が11・10・10と揺らいだため、既定では著者が選んだモデルを同梱して使います。評価では、0.20・0.30 m/sのv6の結果として、同じ評価のv0の走行をそのまま使います（同じ制御器を2回走らせると、GPUのわずかな計算差で際どい条件の結果が入れ替わることがあるため）。GPUでXLAの決定的計算の設定（`--xla_gpu_deterministic_ops`、`--xla_gpu_autotune_level=0`）も試しましたが、この環境では最初の1ステップで数値が発散したため使っていません。

### 解釈上の注意
- 評価は5つの波×3速度の15条件だけなので、1条件の差は小さな差です。検証の30条件の結果と、開始位置をずらした確認も併せて見てください。
- 甲板は40 m×4 mの計算用プラットフォームで、実船の寸法・慣性・波は再現していません。甲板の中心から12 m離れた位置で歩き始めるため、ピッチ（縦揺れ）による上下の加速度が大きくなります。
- シミュレーションの結果であり、実機での転倒防止を保証するものではありません。補正モデルは甲板の姿勢・速度をシミュレーションから直接受け取っており、実機ではセンサーによる推定が必要です。

出典：[Walk These Ways](https://github.com/Improbable-AI/walk-these-ways)（commit 0e7236b）、[MuJoCo Menagerie Go1](https://github.com/google-deepmind/mujoco_menagerie/tree/main/unitree_go1)（commit 0059d43）、[Brax](https://github.com/google/brax)、[MuJoCo MJX](https://mujoco.readthedocs.io/en/stable/mjx.html)。